> **Example run — constructed test case, not a real FAERS report.**
>
> - **When and how:** run in Google Colab on 20 October 2026 with real DeepSeek and Gemini 2.5 Pro calls, using an **earlier version** of the one-case notebook. That version copied the same repository code but set Gemini to temperature 0 and had a `MAX_PAIRS` option. The current notebook is [`../MARCA_one_case.ipynb`](../MARCA_one_case.ipynb); its usage guide is [`../README.md`](../README.md).
> - **The case** was made by editing the repository's example case (FAERS 229933175, icatibant). Drug changed to COUMADIN (warfarin, oral), event to *Gastrointestinal Bleeding*, age to 47 years, indication to *Secondary prevention of MACE*. Several fields were left from the original (case ID, weight 20 kg, NDA 22150, manufacturer, country). Because the case ID belongs to a knowledge-base case, that case's records (icatibant) were excluded from retrieval, and the comparison with the expert scores (cell 23) is empty.
> - **Label file:** Agent 2's label file for COUMADIN was built from the FDA label PDF `https://www.accessdata.fda.gov/drugsatfda_docs/label/2011/009218s107lbl.pdf` (cell 15).
> - **Results** (1 DEC):
>   - Agent 2: *Not listed (Yes)*, so Q1 = +1 (the term is not in the label list word for word).
>   - Agent 3: *Sign*, so Q10 = 1; the RAG Q10 answer was 0.
>   - Agent 4: plausibility HIGH.
>   - RAG: Q2 = 0 and Q5 = 0, with and without references.
>   - Provisional total: 2 (*Possible*), with Q3, Q4 and Q6–Q9 not yet scored by a reviewer.
>
> The outputs are kept as an illustration of what the notebook returns. They are not results of the study.

# MARCA — run one FAERS case end to end

This notebook puts the MARCA Python code into one place and runs it on a single FAERS case. The code comes from the repository's working path: `software/config.py`, `software/case_parsing`, `software/agents/pipeline.py`, `software/rag/__init__.py` and the score parser in `software/evaluation`.

For every drug–event combination (DEC) in the case it runs:

| Step | Model | Output |
|---|---|---|
| Agent 2: label listedness (Q1) | DeepSeek | Listed in label / Not listed (Yes, No, Unclear) |
| Agent 3: sign vs symptom (Q10) | DeepSeek | Sign = 1, Symptom = 0 |
| Agent 1: PubMed retrieval and summary | Gemini 2.5 Pro via CrewAI | Mechanism of action and evidence summary |
| Agent 4: clinical causality report (Q2–Q9) | Gemini 2.5 Pro via CrewAI | Structured report and HIGH / MEDIUM / LOW rating |
| Selective RAG (Q2, Q5, Q10) | DeepSeek + expert knowledge base | Score and reasoning without and with expert reference cases |

**What you need**
- A DeepSeek API key and a Google AI Studio (Gemini) API key. Add them in Colab's **Secrets** panel (key icon on the left) as `DEEPSEEK_API_KEY` and `GOOGLE_API_KEY`, and allow this notebook to read them. If they are missing, the notebook asks for them.
- Nothing else: the expert knowledge base (`expert_kb.xlsx`) and the example case are downloaded from the public GitHub repository.

**Time and cost.** The example case has 10 DECs: about 20 Gemini 2.5 Pro calls and about 80 DeepSeek calls, typically 10–15 minutes. Set `MAX_PAIRS` to 1 or 2 for a first try.

The example case (FAERS 229933175, icatibant) is part of the study's knowledge base, so the last section compares this run with the study's stored MARCA scores and with the expert gold standard. Its own records are excluded from retrieval, as in the study.

Agents 1 and 4 run with `verbose=True`, as in the repository, so the run cell prints a lot of CrewAI output.

Run the cells from top to bottom (**Runtime → Run all** works once the keys are set).

### What differs from the repository code

The agent, RAG and parsing code below is the repository's code with only these changes:

1. **Gemini temperature.** The repository creates the CrewAI agents with `llm="gemini/gemini-2.5-pro"` and no temperature, so Gemini runs at the provider default. The manuscript says all models ran at temperature 0. Here `GEMINI_TEMPERATURE = 0` by default; set it to `None` to reproduce the repository behaviour exactly.
2. **`MAX_PAIRS`.** Optional limit on how many DECs go through Agents 1 + 4 and the RAG step, for quick tests. `0` means all.
3. **Score sheet (last section).** Not in the repository. It collects what the agents return into a provisional Naranjo Q1–Q10 sheet. As in the manuscript (Section 2.4.5), Q3, Q4 and Q6–Q9 are meant to be scored by a person reading Agent 4's report, so they start as placeholders you can override.
4. **Paths and keys.** Files live under `/content/marca`; keys come from Colab Secrets instead of a `.env` file.

The modular sub-packages in the repository (`software/agents/drug_label_assessment`, `objective_evidence_assessment`, `scientific_evidence_retrieval`, `drug_class_resolution`) import a `software.llm` module that is not in the repository, so they are not used here.

## 1. Setup

In [4]:
#@title 1. Install packages
# If Colab asks you to restart the session after this cell, restart and continue from cell 2.
%pip install -q "crewai==0.126.0" openpyxl scikit-learn rich PyPDF2 pymupdf


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.9/157.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.

In [5]:
#@title 2. Settings
#@markdown **Agents 1 + 4** (PubMed summary and clinical causality report, Gemini via CrewAI)
RUN_CREW = True  #@param {type:"boolean"}
#@markdown **Selective RAG** for Q2, Q5 and Q10 (DeepSeek + expert knowledge base)
USE_RAG = True  #@param {type:"boolean"}
#@markdown Resolve drug classes with openFDA → RxNorm → DeepSeek for class-level retrieval (about 3–5 minutes for the 195 KB drugs; untick to skip the class tier)
RESOLVE_DRUG_CLASSES = True  #@param {type:"boolean"}
#@markdown Models
GEMINI_MODEL = "gemini/gemini-2.5-pro"  #@param {type:"string"}
GEMINI_TEMPERATURE = 0  #@param {type:"raw"}
DEEPSEEK_MODEL = "deepseek-chat"  #@param {type:"string"}
#@markdown Limit the number of DECs sent to Agents 1 + 4 and RAG (0 = all)
MAX_PAIRS = 0  #@param {type:"integer"}

# Public repository with the expert knowledge base and the example case
GITHUB_RAW = ("https://raw.githubusercontent.com/daisyraelopez/"
              "Multi-Agent-Retrieval-Augmented-for-Individual-level-Causality-Assessment-"
              "MARCA-in-Pharmacovigilance/main")


In [18]:
#@title 3. API keys (Colab Secrets or prompt)
import os
from getpass import getpass


def get_secret(name: str) -> str:
    """Read a key from Colab Secrets, then the environment, then ask for it."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    if os.getenv(name):
        return os.environ[name]
    return getpass(f"{name}: ")


os.environ["DEEPSEEK_API_KEY"] = get_secret("DEEPSEEK_API_KEY")
if RUN_CREW:
    os.environ["GOOGLE_API_KEY"] = get_secret("GOOGLE_API_KEY")
    # CrewAI/LiteLLM's Gemini provider reads GEMINI_API_KEY (same as ensure_gemini_env() in the repo)
    os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]

# Keep CrewAI quiet: no telemetry
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
print("Keys set:", [k for k in ("DEEPSEEK_API_KEY", "GEMINI_API_KEY") if os.getenv(k)])


DEEPSEEK_API_KEY: ··········
GOOGLE_API_KEY: ··········
Keys set: ['DEEPSEEK_API_KEY', 'GEMINI_API_KEY']


## 2. MARCA code from the repository
These cells only define functions. Click **Show code** on a cell to read it.

In [6]:
#@title 4. Configuration and DeepSeek call (software/config.py)
import json
import os
import types
from pathlib import Path

import requests

ROOT_DIR = Path("/content/marca") if Path("/content").exists() else Path.cwd() / "marca"
DATA_DIR = ROOT_DIR / "data"
KB_DIR = DATA_DIR / "knowledge_base"
OUTPUT_DIR = ROOT_DIR / "output"

config = types.SimpleNamespace(
    ROOT_DIR=ROOT_DIR,
    RAW_DIR=DATA_DIR / "raw",
    KB_DIR=KB_DIR,
    KB_PATH=KB_DIR / "expert_kb.xlsx",                    # expert-assessed DECs
    DRUG_CLASS_CACHE_PATH=KB_DIR / "drug_class_map.json",  # cached drug -> class map
    LABELS_DIR=KB_DIR / "drug_labels",                    # one AE-list CSV per drug
    RESULTS_DIR=OUTPUT_DIR / "results",
    FIGURES_DIR=OUTPUT_DIR / "figures",
    LOGS_DIR=OUTPUT_DIR / "logs",
    # Models (DeepSeek always at temperature 0, as in the repository)
    DEEPSEEK_API_URL="https://api.deepseek.com/chat/completions",
    DEEPSEEK_MODEL=DEEPSEEK_MODEL,
    CREW_LLM=GEMINI_MODEL,
    TEMPERATURE=0,
    RUN_CREW=RUN_CREW,
    # Context limits used in the RAG prompts
    CASE_CONTEXT_CHARS=1400,
    LITERATURE_CONTEXT_CHARS=800,
)


def ensure_dirs():
    """Create the output folders if they do not exist."""
    for d in (config.RESULTS_DIR, config.FIGURES_DIR, config.LOGS_DIR, config.LABELS_DIR, config.RAW_DIR):
        d.mkdir(parents=True, exist_ok=True)


config.ensure_dirs = ensure_dirs
ensure_dirs()


def require_key(name: str) -> str:
    value = os.getenv(name, "")
    if not value:
        raise EnvironmentError(f"{name} is not set. Run the API-keys cell first.")
    return value


def crew_llm():
    """LLM handed to the CrewAI agents (Agents 1 and 4).

    GEMINI_TEMPERATURE = None reproduces the repository exactly (model string, provider
    default temperature). Any number passes that temperature to Gemini.
    """
    if GEMINI_TEMPERATURE is None:
        return config.CREW_LLM
    from crewai import LLM
    return LLM(model=config.CREW_LLM, temperature=float(GEMINI_TEMPERATURE))


DEFAULT_SYSTEM = (
    "You are a senior pharmacovigilance expert performing drug causality "
    "assessment. Answer concisely with a numeric score and a brief justification."
)


def call_deepseek(prompt: str, system: str = DEFAULT_SYSTEM, json_mode: bool = False,
                  timeout: int = 60) -> str:
    """Send one chat request to DeepSeek (temperature 0) and return the text reply."""
    key = require_key("DEEPSEEK_API_KEY")
    payload = {
        "model": config.DEEPSEEK_MODEL,
        "temperature": config.TEMPERATURE,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
    }
    if json_mode:
        payload["response_format"] = {"type": "json_object"}
    r = requests.post(
        config.DEEPSEEK_API_URL,
        headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
        json=payload,
        timeout=timeout,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"].strip()


def call_deepseek_json(prompt: str, system: str, timeout: int = 60) -> dict:
    """Call DeepSeek in JSON mode and parse the reply."""
    return json.loads(call_deepseek(prompt, system=system, json_mode=True, timeout=timeout))


print("Working folder:", ROOT_DIR)


Working folder: /content/marca


In [7]:
#@title 5. Case parsing (software/case_parsing, unchanged)
"""FAERS case parsing: drug-record deduplication and structured narrative -> DataFrames."""
import re

import pandas as pd
from rich.console import Console

console = Console()


# ════════════════════════════════════════════════════════════════════════════
# Deduplication
# ════════════════════════════════════════════════════════════════════════════
import re


def deduplicate_drugs(text):
    """Keep one record per drug (the one with the most non-missing fields) and
    return the case as one "Field: value" line per field, ready for `process_case`."""
    if not text.strip():
        return "The input box is empty."

    # 1. Define the full list of keys to find in your text
    all_keys = [
        'Case ID', 'Case Version', 'FDA Code', 'Event Date', 'Manufacture Date',
        'Initial FDA Date', 'FDA Date', 'Report Code', 'Authorization Number',
        'Manufacturer Number', 'Manufacturer Sender', 'Literature Reference',
        'Age', 'Age Code', 'Age Group', 'Sex', 'E Sub', 'Weight', 'Weight Code',
        'Report Date', 'To Manufacturer', 'Occupation Code', 'Reporter Country',
        'Occurrence Country', 'Adverse Reaction(s)', 'Case ID Number',
        'Drug Sequence', 'Role Code', 'Drug Name', 'Product AI', 'Val VBM',
        'Route', 'Dose VBM', 'Cumulative Dose (Char)', 'Cumulative Dose (Unit)',
        'Dechal', 'Rechal', 'Lot Number', 'Expiration Date', 'NDA Number',
        'Dose Amount', 'Dose Unit', 'Dose Form', 'Dose Frequency',
        'Indication for Use', 'Therapy Start Date', 'Therapy End Date',
        'Duration', 'Duration Code'
    ]

    # 2. Extract values using Key Anchors
    pattern = '|'.join([re.escape(k) for k in all_keys])
    matches = list(re.finditer(f'({pattern}):', text))

    data_dict = {}
    for i in range(len(matches)):
        start_idx = matches[i].end()
        end_idx = matches[i+1].start() if i+1 < len(matches) else len(text)
        key = matches[i].group(1)
        value = text[start_idx:end_idx].strip()
        data_dict[key] = value

    if 'Drug Name' not in data_dict:
        return "Error: Could not find 'Drug Name'. Ensure the input has 'Drug Name:'"

    # 3. List of fields that are "aligned" (semicolon separated)
    aligned_fields = [
        'Drug Sequence', 'Role Code', 'Drug Name', 'Product AI', 'Val VBM',
        'Route', 'Dose VBM', 'Cumulative Dose (Char)', 'Cumulative Dose (Unit)',
        'Dechal', 'Rechal', 'Lot Number', 'Expiration Date', 'NDA Number',
        'Dose Amount', 'Dose Unit', 'Dose Form', 'Dose Frequency',
        'Indication for Use', 'Therapy Start Date', 'Therapy End Date',
        'Duration', 'Duration Code'
    ]

    # 4. Group by Drug Name and pick the best index
    drug_names = [d.strip() for d in data_dict['Drug Name'].split(';')]
    best_indices = {} # {drug_name: index}

    for i, name in enumerate(drug_names):
        # Calculate info score for this specific index
        current_score = 0
        for field in aligned_fields:
            vals = [v.strip() for v in data_dict.get(field, "").split(';')]
            if i < len(vals) and vals[i].upper() != "NA" and vals[i] != "":
                current_score += 1

        # If we haven't seen this drug, or this version is better, save index
        if name not in best_indices:
            best_indices[name] = (i, current_score)
        else:
            if current_score > best_indices[name][1]:
                best_indices[name] = (i, current_score)

    # Sorted list of the original indices we want to keep
    target_indices = sorted([val[0] for val in best_indices.values()])

    # 5. Reconstruct the text structure
    output_parts = []
    for key in all_keys:
        if key in data_dict:
            if key in aligned_fields:
                # Filter values to only keep the "best" indices
                raw_vals = [v.strip() for v in data_dict[key].split(';')]
                # Ensure we have at least NA if index is missing
                filtered_vals = []
                for idx in target_indices:
                    if idx < len(raw_vals):
                        v = raw_vals[idx]
                        filtered_vals.append(v if v != "" else "NA")
                    else:
                        filtered_vals.append("NA")

                new_value = " ; ".join(filtered_vals)
                output_parts.append(f"{key}: {new_value}")
            else:
                # Static fields (Age, Case ID, etc.) stay as they are
                output_parts.append(f"{key}: {data_dict[key]}")

    return " \n".join(output_parts)


# Name used in the original notebook
process_case_text_to_text = deduplicate_drugs


# ════════════════════════════════════════════════════════════════════════════
# Parser
# ════════════════════════════════════════════════════════════════════════════


def parse_case_text(case_text):
    lines = case_text.strip().split("\n")
    data = {}

    for line in lines:
        line = line.strip()
        if ":" in line:
            key, value = line.split(":", 1)
            data[key.strip()] = value.strip()

    case_info = {
        "Case ID": data.get("Case ID", "NA"),
        "Case Version": data.get("Case Version", "NA"),
        "FDA Code": data.get("FDA Code", "NA"),
        "Event Date": data.get("Event Date", "NA"),
        "Manufacture Date": data.get("Manufacture Date", "NA"),
        "Initial FDA Date": data.get("Initial FDA Date", "NA"),
        "FDA Date": data.get("FDA Date", "NA"),
        "Report Code": data.get("Report Code", "NA"),
        "Authorization Number": data.get("Authorization Number", "NA"),
        "Manufacturer Number": data.get("Manufacturer Number", "NA"),
        "Manufacturer Sender": data.get("Manufacturer Sender", "NA"),
        "Literature Reference": data.get("Literature Reference", "NA"),
        "Age": data.get("Age", "NA"),
        "Age Code": data.get("Age Code", "NA"),
        "Age Group": data.get("Age Group", "NA"),
        "Sex": data.get("Sex", "NA"),
        "E Sub": data.get("E Sub", "NA"),
        "Weight": data.get("Weight", "NA"),
        "Weight Code": data.get("Weight Code", "NA"),
        "Report Date": data.get("Report Date", "NA"),
        "To Manufacturer": data.get("To Manufacturer", "NA"),
        "Occupation Code": data.get("Occupation Code", "NA"),
        "Reporter Country": data.get("Reporter Country", "NA"),
        "Occurrence Country": data.get("Occurrence Country", "NA"),
        "Case ID Number": data.get("Case ID Number", "NA"),
        "EDSS": data.get("EDSS", "NA"),
    }

    adverse_reactions_raw = data.get("Adverse Reaction(s)", "")
    adverse_reactions = [r.strip() for r in adverse_reactions_raw.split(";") if r.strip()]

    drug_fields = [
        "Drug Sequence", "Role Code", "Drug Name", "Product AI", "Val VBM",
        "Route", "Dose VBM", "Cumulative Dose (Char)", "Cumulative Dose (Unit)",
        "Dechal", "Rechal", "Lot Number", "Expiration Date", "NDA Number",
        "Dose Amount", "Dose Unit", "Dose Form", "Dose Frequency",
        "Indication for Use", "Therapy Start Date", "Therapy End Date",
        "Duration", "Duration Code"
    ]

    drug_data = {}

    if "Drug Sequence" in data:
        sequences = [v.strip() for v in data["Drug Sequence"].split(";")]
        max_drugs = len(sequences)
        drug_data["Drug Sequence"] = sequences
    else:
        max_drugs = 0

    for field in drug_fields:
        if field == "Drug Sequence":
            continue

        if field in data:
            values = [v.strip() for v in data[field].split(";")]

            if field == "Indication for Use" and len(values) > max_drugs:
                values = values[:max_drugs]

            while len(values) < max_drugs:
                values.append("NA")

            drug_data[field] = values[:max_drugs]

    drugs_list = []
    for i in range(max_drugs):
        drug = {}
        for field in drug_fields:
            if field in drug_data and i < len(drug_data[field]):
                drug[field] = drug_data[field][i]
            else:
                drug[field] = "NA"
        drugs_list.append(drug)

    return case_info, adverse_reactions, drugs_list


def create_dataframes_from_case(case_info, adverse_reactions, drugs_list):
    df_case = pd.DataFrame([case_info])

    if adverse_reactions:
        df_reactions = pd.DataFrame({
            "Case ID": [case_info["Case ID"]] * len(adverse_reactions),
            "Adverse Reaction": adverse_reactions
        })
    else:
        df_reactions = pd.DataFrame(columns=["Case ID", "Adverse Reaction"])

    if drugs_list:
        df_drugs = pd.DataFrame(drugs_list)
        df_drugs.insert(0, "Case ID", case_info["Case ID"])
    else:
        df_drugs = pd.DataFrame()

    if not df_drugs.empty:
        df_comprehensive = df_drugs.copy()
        for key, value in case_info.items():
            if key not in df_comprehensive.columns:
                df_comprehensive[key] = value
    else:
        df_comprehensive = pd.DataFrame()

    return df_case, df_reactions, df_drugs, df_comprehensive


def process_case(case_text, display=True):
    case_info, adverse_reactions, drugs_list = parse_case_text(case_text)
    df_case, df_reactions, df_drugs, df_comprehensive = create_dataframes_from_case(
        case_info, adverse_reactions, drugs_list
    )

    if display:
        console.print("\n" + "=" * 80, style="bold cyan")
        console.print("FDA CASE PARSED INTO DATAFRAMES", style="bold cyan")
        console.print("=" * 80, style="bold cyan")
        console.print(f"\n✅ Case ID: {case_info['Case ID']}", style="green")
        console.print(f"✅ Drugs: {len(drugs_list)}", style="green")
        console.print(f"✅ Adverse Reactions: {len(adverse_reactions)}", style="green")

    return df_case, df_reactions, df_drugs, df_comprehensive


In [8]:
#@title 6. Agent 1 — Scientific evidence retrieval (software/agents/pipeline.py)
import asyncio
import csv
import os
import re
import tempfile
import time
from pathlib import Path
from xml.etree import ElementTree as ET

import pandas as pd
import requests
from rich.console import Console


console = Console()

try:
    from crewai import Agent, Task
except ImportError:      # CrewAI is only needed for Agents 1 and 4
    Agent = Task = None

# ════════════════════════════════════════════════════════════════════════════
# Agent 1 — Scientific Evidence Retrieval (PubMed + summary)
# ════════════════════════════════════════════════════════════════════════════





def safe_filename(name):
    return str(name).replace(" ", "_").replace("/", "-").replace(":", "")

def search_pubmed(drug_name, adr_term, max_results=3):
    query = f"{drug_name} AND {adr_term} adverse event"
    console.print(f"\n🔍 Searching PubMed for: '{query}'", style="bold yellow")

    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    search_params = {"db": "pubmed", "term": query, "retmax": max_results, "retmode": "json"}

    try:
        search_resp = requests.get(search_url, params=search_params, timeout=30)
        search_resp.raise_for_status()
        ids = search_resp.json().get("esearchresult", {}).get("idlist", [])
    except Exception as e:
        console.print(f"❌ PubMed search failed: {e}", style="red")
        return []

    if not ids:
        return []

    fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    fetch_params = {"db": "pubmed", "id": ",".join(ids), "retmode": "xml"}

    try:
        fetch_resp = requests.get(fetch_url, params=fetch_params, timeout=30)
        fetch_resp.raise_for_status()
        root = ET.fromstring(fetch_resp.text)
    except Exception as e:
        console.print(f"❌ PubMed fetch failed: {e}", style="red")
        return []

    results = []
    for article in root.findall(".//PubmedArticle"):
        title = article.findtext(".//ArticleTitle")
        abstract_parts = [t.text for t in article.findall(".//AbstractText") if t.text]
        abstract = " ".join(abstract_parts) if abstract_parts else "No Abstract Available."
        pmid = article.findtext(".//PMID")
        results.append({"pmid": pmid, "title": title, "abstract": abstract})

    return results

def save_literature_to_file(drug_name, adr_term, evidence, assessment_type="pubmed", out_dir=None):
    out_dir = Path(out_dir or config.LOGS_DIR / "literature")
    out_dir.mkdir(parents=True, exist_ok=True)
    filename = str(out_dir / f"{safe_filename(drug_name)}_{safe_filename(adr_term)}_{assessment_type}_results.txt")
    output_lines = [
        f"--- Literature Search Results for: {drug_name} AND {adr_term} ---",
        f"Date Generated: {time.ctime()}",
        f"Articles Found: {len(evidence)}"
    ]

    for i, paper in enumerate(evidence, 1):
        output_lines.append(f"\n{'=' * 50}")
        output_lines.append(f"ARTICLE {i}")
        output_lines.append(f"PMID: {paper.get('pmid', 'N/A')}")
        output_lines.append(f"Title: {paper.get('title', 'N/A')}")
        output_lines.append(f"Abstract:\n{paper.get('abstract', 'No Abstract Available.')}")

    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(output_lines))

    return filename, "\n".join(output_lines)




def build_literature_agent(drug: str, event: str) -> Agent:
    return Agent(
        role="Expert Medical Searcher and Literature Summarizer",
        goal=f"Analyze PubMed results for '{drug}' and '{event}' to summarize MOA and clinical evidence.",
        backstory="Detailed researcher.",
        llm=crew_llm(),  # notebook: was llm=config.CREW_LLM
        verbose=True,
    )


def build_literature_task(agent: Agent, literature_content: str) -> Task:
    return Task(
        description=f"Summarize PubMed content:\n{literature_content}",
        agent=agent,
        expected_output="MOA & Clinical Evidence Summary",
    )


In [9]:
#@title 7. Agent 2 — Drug label assessment, Naranjo Q1 (software/agents/pipeline.py)
# ════════════════════════════════════════════════════════════════════════════
# Agent 2 — Drug Label Assessment (Naranjo Q1)
# ════════════════════════════════════════════════════════════════════════════


LABEL_SYSTEM = "You are a medical documentation assistant that evaluates AE-drug relations."


def load_ae_list_for_drug(drug_name, labels_dir=None):
    """Load the AE list CSV whose file name contains the drug name (None if there is no file)."""
    search_dir = str(labels_dir or config.LABELS_DIR)
    # compare letters and digits only, so "A\\B" matches the file "A_B.csv"
    drug_search = re.sub(r"[^a-z0-9]", "", drug_name.lower())
    csv_files = sorted(f for f in os.listdir(search_dir) if f.lower().endswith(".csv"))
    base = lambda f: re.sub(r"[^a-z0-9]", "", os.path.splitext(f)[0].lower())
    # exact file-name match first (e.g. XELJANZ.csv before XELJANZ_XR.csv), then partial match
    ordered = [f for f in csv_files if base(f) == drug_search] + \
              [f for f in csv_files if base(f) != drug_search and drug_search in base(f)]
    for file in ordered:
            full_path = os.path.join(search_dir, file)
            with open(full_path, "r", encoding="utf-8") as f:
                reader = csv.reader(f)
                next(reader, None)  # skip header (product name)
                return [row[0].strip().lower() for row in reader if row]
    return None   # no label file for this drug


def clean_name(name):
    """Remove unwanted substrings and extra spaces."""
    if not name:
        return ""
    name = name.strip()
    for remove_str in ["Val", "Drug Record"]:
        name = name.replace(remove_str, "")
    return name.strip()


def assess_pair(drug: str, ae: str, label_aes: list) -> dict:
    """Listedness for one drug-event pair."""
    if ae.lower() in label_aes:
        return {"drug": drug, "ae": ae, "status": "Listed in label", "explanation": ""}

    prompt = f"""
Determine if the following adverse event is medically related to the drug based on clinical reasoning.

Drug: {drug}
Adverse Event: {ae}

Output in this format:
Related - start: <Yes/No/Unclear> end.
Explanation - start: <Brief medical explanation> end.
"""
    try:
        response = call_deepseek(prompt, system=LABEL_SYSTEM)
        rel_match = re.search(r"Related - start:\s*(.+?)\s*end\.", response, re.DOTALL)
        relation = rel_match.group(1).strip() if rel_match else "Unclear"
        exp_match = re.search(r"Explanation - start:\s*(.+?)\s*end\.", response, re.DOTALL)
        explanation = exp_match.group(1).strip() if exp_match else response
        return {"drug": drug, "ae": ae, "status": f"Not listed ({relation})", "explanation": explanation}
    except Exception as e:
        return {"drug": drug, "ae": ae, "status": "Error during DeepSeek call", "explanation": str(e)}


def assess_listedness(drugs: list, adverse_events: list, labels_dir=None) -> list:
    """Q1 results for every drug x AE pair. Drugs without a label file are skipped."""
    drugs = list(dict.fromkeys(clean_name(d) for d in drugs if d and d.strip()))
    adverse_events = list(dict.fromkeys(clean_name(a) for a in adverse_events if a and a.strip()))
    results = []
    for drug in drugs:
        label_aes = load_ae_list_for_drug(drug, labels_dir)
        if label_aes is None:
            print(f"⚠️ No label CSV found for {drug}. Skipping.")
            continue
        label_aes = [x.lower() for x in label_aes]
        for ae in adverse_events:
            results.append(assess_pair(drug, ae, label_aes))
    return results


In [10]:
#@title 8. Agent 2 — Building the per-drug label files (software/agents/pipeline.py)
# ════════════════════════════════════════════════════════════════════════════
# Agent 2 — building the per-drug label files
# ════════════════════════════════════════════════════════════════════════════




# ---------------------------
# PDF download
# ---------------------------
def download_pdf(url, output_path):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/115.0.0.0 Safari/537.36"
        )
    }
    with requests.get(url, headers=headers, timeout=30, stream=True) as r:
        r.raise_for_status()
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)


# ---------------------------
# Text extraction (PyPDF2 primary, pymupdf fallback)
# ---------------------------
def extract_text(pdf_path):
    import PyPDF2
    full_text = ""
    try:
        with open(pdf_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            page_texts = [page.extract_text() for page in reader.pages if page.extract_text()]
            full_text = "\n".join(page_texts)
    except Exception as e:
        print(f"⚠️ PyPDF2 extraction error: {e}")

    if len(full_text.strip()) < 200:
        try:
            import fitz
            doc = fitz.open(pdf_path)
            full_text = ""
            for page in doc:
                full_text += page.get_text("text") or ""
            doc.close()
            print("ℹ️ Used pymupdf fallback for text extraction.")
        except Exception as e:
            print(f"⚠️ pymupdf extraction failed: {e}")

    return full_text


# ---------------------------
# DeepSeek steps
# ---------------------------
def extract_adverse_section(full_text):
    """Return only the label's adverse events / adverse reactions section."""
    prompt = (
        "Extract ONLY the full text section from the following drug label text "
        "that discusses 'Adverse Events', 'Adverse Reactions', or similar safety information. "
        "Do not summarize, rewrite, or add commentary. "
        "Return only the raw text of that section."
    )
    return call_deepseek(f"{prompt}\n\n---\n\n{full_text}",
                         system="You are a precise document parser.", timeout=300)


def extract_adverse_events(section_text):
    """Return the list of adverse events mentioned in the section text."""
    prompt = f"""
    You are given the undesirable effects section of a drug product.
    Extract ONLY the list of adverse events mentioned in the text.
    Each adverse event must be listed separately, without duplicates.
    Do not include any explanatory text, frequencies, or formatting, only the adverse events.
    Output them as a plain list, one per line.

    Text:
    {section_text}
    """
    ae_text = call_deepseek(prompt, system="You are an expert assistant that extracts structured data.",
                            timeout=300)
    return [line.strip() for line in ae_text.split("\n") if line.strip()]


def save_to_csv(product_name, adverse_events, output_file):
    """Save the extracted adverse events (first row = product name)."""
    with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow([product_name])
        for ae in adverse_events:
            writer.writerow([ae])
    print(f"✅ CSV file saved: {output_file}")


def build_label_csv(pdf_url_or_path: str, product_name: str, out_dir=None) -> Path:
    """Full pipeline: label PDF -> adverse-event CSV named after the product."""
    out_dir = Path(out_dir or config.LABELS_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory() as tmpdir:
        if os.path.exists(pdf_url_or_path):
            pdf_path = pdf_url_or_path
        else:
            pdf_path = os.path.join(tmpdir, "file.pdf")
            print(f"📥 Downloading: {pdf_url_or_path}")
            download_pdf(pdf_url_or_path, pdf_path)
        text = extract_text(pdf_path)
    if not text.strip():
        raise ValueError("No text extracted from PDF.")
    section = extract_adverse_section(text)
    aes = extract_adverse_events(section)
    output_file = out_dir / f"{product_name.upper()}.csv"
    save_to_csv(product_name.upper(), aes, output_file)
    return output_file


def safe_label_name(drug: str) -> str:
    """File-system-safe name for a drug (combination products use '_' instead of '\\')."""
    import re
    return re.sub(r"[^A-Za-z0-9]+", "_", drug.strip().upper()).strip("_")


def build_all_label_csvs(sources_csv=None, out_dir=None, overwrite=False):
    """Rebuild every drug's label CSV from data/knowledge_base/drug_label_sources.csv.

    The sources file lists the FDA label PDF used for each drug in the study (taken from
    the Q1 rationale in expert_kb.xlsx). Drugs with more than one label get the union of
    the adverse events from all their labels. Requires DEEPSEEK_API_KEY and internet access.
    """
    import pandas as pd
    sources_csv = Path(sources_csv or config.KB_DIR / "drug_label_sources.csv")
    out_dir = Path(out_dir or config.LABELS_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    src = pd.read_csv(sources_csv)
    done, failed = [], []
    for drug, rows in src.groupby("drug"):
        name = safe_label_name(drug)
        target = out_dir / f"{name}.csv"
        if target.exists() and not overwrite:
            continue
        aes = []
        try:
            for url in rows["fda_label_pdf"]:
                with tempfile.TemporaryDirectory() as tmpdir:
                    pdf_path = os.path.join(tmpdir, "label.pdf")
                    download_pdf(url, pdf_path)
                    text = extract_text(pdf_path)
                if text.strip():
                    aes += extract_adverse_events(extract_adverse_section(text))
            aes = list(dict.fromkeys(a for a in aes if a))
            save_to_csv(name, aes, target)
            done.append(drug)
        except Exception as e:
            print(f"❌ {drug}: {e}")
            failed.append(drug)
    print(f"✅ Built {len(done)} label files; {len(failed)} failed.")
    return done, failed


def labels_from_study_workbook(kb_path=None, out_dir=None):
    """Recreate the drug label files from the study's recorded Agent 2 output.

    `q1_ai_answer` in expert_kb.xlsx stores Agent 2's result for every DEC. Events with
    status "Listed in label" were exact matches in the original label file of that drug,
    so they are written to `<DRUG>.csv`. A file is written for every drug (some with no
    events), so Agent 2 assesses every DEC exactly as in the study: listed events are
    matched directly and all others go to DeepSeek.

    These files contain only the label events that occurred in the study's DECs, not the
    complete label. For complete lists use `build_all_label_csvs()`.
    """
    import re
    import pandas as pd
    kb_path = kb_path or config.KB_PATH
    out_dir = Path(out_dir or config.LABELS_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_excel(kb_path, sheet_name=0, usecols=["drug_name", "pt", "q1_ai_answer"])
    df["drug"] = df["drug_name"].astype(str).str.strip()
    df["ae"] = df["pt"].astype(str).str.strip()
    df["status"] = df["q1_ai_answer"].astype(str).str.extract(r"status:\s*([^\n]+)")[0].str.strip()
    n_events = 0
    for drug, rows in df.groupby("drug"):
        listed = rows.loc[rows["status"] == "Listed in label", "ae"]
        listed = list(dict.fromkeys(listed))
        save_to_csv(safe_label_name(drug), listed, out_dir / f"{safe_label_name(drug)}.csv")
        n_events += len(listed)
    print(f"✅ {df['drug'].nunique()} label files, {n_events} listed events, from {kb_path}")
    return out_dir


In [11]:
#@title 9. Agent 3 — Objective evidence assessment, Naranjo Q10 (software/agents/pipeline.py)
# ════════════════════════════════════════════════════════════════════════════
# Agent 3 — Objective Evidence Assessment (Naranjo Q10)
# ════════════════════════════════════════════════════════════════════════════


SIGN_SYMPTOM_SYSTEM = (
    "You are a clinical pharmacovigilance expert. "
    "Classify events as 'Sign' (Value 1) or 'Symptom' (Value 0)."
)


def unique_events(adverse_events: list) -> list:
    """Remove duplicate AE terms (case-insensitive), keeping their order."""
    seen, out = set(), []
    for ae in adverse_events:
        key = ae.strip().lower()
        if key and key not in seen:
            seen.add(key)
            out.append(ae.strip())
    return out


def classify_aes(aes: list, pause: float = 0.2) -> list:
    results = []
    for ae in unique_events(aes):
        print(f"Analyzing: {ae}")
        prompt = f"""
Analyze the medical term: "{ae}"

Classify as:
- SIGN: Objective evidence (observable/measurable). Value = 1.
- SYMPTOM: Subjective experience (reported by patient). Value = 0.
- SYMPTOM: Psychiatric disorders are all symptoms. Value = 0.

Return JSON:
{{
  "ae": "{ae}",
  "classification": "Sign" or "Symptom",
  "value": 1 or 0,
  "rationale": "Brief medical explanation"
}}
"""
        try:
            results.append(call_deepseek_json(prompt, system=SIGN_SYMPTOM_SYSTEM))
        except Exception as e:
            print(f"Error analyzing {ae}: {e}")
            results.append({"ae": ae, "classification": None, "value": None, "rationale": str(e)})
        time.sleep(pause)
    return results


In [12]:
#@title 10. Agent 4 — Clinical causality assessment (software/agents/pipeline.py)
# ════════════════════════════════════════════════════════════════════════════
# Agent 4 — Clinical Causality Assessment
# ════════════════════════════════════════════════════════════════════════════





def _get_value_or_missing(value):
    if pd.isna(value):
        return "MISSING"
    str_val = str(value).strip()
    return "MISSING" if str_val.upper() in ("NA", "UNK", "N/A", "") else str_val

def _extract_case_metadata(df_case):
    if df_case.empty:
        return {
            "age": "MISSING",
            "sex": "MISSING",
            "weight": "MISSING",
            "history_similar_reaction": "MISSING",
            "outcome": "MISSING",
            "reporter_country": "MISSING",
            "edss": "MISSING",
        }

    row = df_case.iloc[0]
    return {
        "age": _get_value_or_missing(row.get("Age", "MISSING")),
        "sex": _get_value_or_missing(row.get("Sex", "MISSING")),
        "weight": _get_value_or_missing(row.get("Weight", "MISSING")),
        "history_similar_reaction": _get_value_or_missing(row.get("MISSING", "MISSING")),
        "outcome": _get_value_or_missing(row.get("Outcome Code", "MISSING")),
        "reporter_country": _get_value_or_missing(row.get("Reporter Country", "MISSING")),
        "edss": _get_value_or_missing(row.get("EDSS", "MISSING")),
    }

def _extract_drug_causality_map(df_drugs):
    causality_map = {}
    if df_drugs.empty:
        return causality_map

    for _, row in df_drugs.iterrows():
        drug_name = _get_value_or_missing(row.get("Drug Name", "MISSING"))
        if drug_name == "MISSING":
            continue

        causality_map[drug_name] = {
            "start": _get_value_or_missing(row.get("Therapy Start Date", "MISSING")),
            "end": _get_value_or_missing(row.get("Therapy End Date", "MISSING")),
            "dechal": _get_value_or_missing(row.get("Dechal", "MISSING")),
            "rechal": _get_value_or_missing(row.get("Rechal", "MISSING")),
            "route": _get_value_or_missing(row.get("Route", "MISSING")),
            "dose": _get_value_or_missing(row.get("Dose VBM", "MISSING")),
            "indication": _get_value_or_missing(row.get("Indication for Use", "MISSING")),
        }

    return causality_map

def _parse_drugs_and_events(df_drugs, df_reactions):
    unique_drugs = []
    unique_events = []

    if not df_drugs.empty and "Drug Name" in df_drugs.columns:
        unique_drugs = df_drugs["Drug Name"].dropna().unique().tolist()
        unique_drugs = [d for d in unique_drugs if str(d).upper() not in ("NA", "UNK", "N/A", "")]

    if not df_reactions.empty and "Adverse Reaction" in df_reactions.columns:
        unique_events = df_reactions["Adverse Reaction"].dropna().unique().tolist()
        unique_events = [e for e in unique_events if str(e).upper() not in ("NA", "UNK", "N/A", "")]

    drug_event_pairs = [(d, e) for d in unique_drugs for e in unique_events if d and e]
    console.print(
        f"✅ Found {len(unique_drugs)} drugs and {len(unique_events)} events from dataframes.",
        style="green"
    )
    return drug_event_pairs


def build_case_context(drug, case_metadata, drug_causality_map, case_event_date):
    """Build the temporal/causality context text for one drug (as in the notebook)."""
    drug_factors = drug_causality_map.get(drug, {})
    concurrent_drugs_list = []
    for concurrent_drug, factors in drug_causality_map.items():
        if concurrent_drug != drug:
            concurrent_drugs_list.append(
                f"- **Drug Name:** {concurrent_drug}\n"
                f"  - **Indication:** {factors.get('indication', 'MISSING')}\n"
                f"  - **Therapy Period:** {factors.get('start', 'MISSING')} to {factors.get('end', 'MISSING')}\n"
                f"  - **Dechal/Rechal:** Dechal: {factors.get('dechal', 'MISSING')}, Rechal: {factors.get('rechal', 'MISSING')}"
            )
    concurrent_medication_context = "\n".join(concurrent_drugs_list) or "N/A: single drug"

    return f"""
**PATIENT AND CASE METADATA:**
- Age: {case_metadata['age']}
- Sex: {case_metadata['sex']}
- Weight: {case_metadata['weight']}
- Outcome Code: {case_metadata['outcome']}
- History of Similar Reactions: {case_metadata['history_similar_reaction']}
- Reporter Country: {case_metadata['reporter_country']}
- EDSS: {case_metadata['edss']}

**DRUG-SPECIFIC INFO:**
- Event Date: {case_event_date}
- Therapy Start: {drug_factors.get('start', 'MISSING')}
- Therapy End: {drug_factors.get('end', 'MISSING')}
- Route: {drug_factors.get('route', 'MISSING')}
- Dose: {drug_factors.get('dose', 'MISSING')}
- Indication: {drug_factors.get('indication', 'MISSING')}
- Dechallenge Result: {drug_factors.get('dechal', 'MISSING')}
- Rechallenge Result: {drug_factors.get('rechal', 'MISSING')}

**Concurrent Medications:**
{concurrent_medication_context}
"""


def build_safety_assessor(drug: str, event: str) -> Agent:
    return Agent(
        role="Pharmacovigilance Causality Specialist",
        goal=f"Assess causality for '{event}' caused by '{drug}' using metadata and literature summary.",
        backstory="Experienced safety specialist.",
        llm=crew_llm(),  # notebook: was llm=config.CREW_LLM
        verbose=True,
    )


def build_assessment_task(safety_assessor: Agent, drug: str, event: str,
                          temporal_causality_context: str) -> Task:
    return Task(
        description=f"""
Use the following context to provide a comprehensive causality assessment for '{drug}' causing '{event}':

{temporal_causality_context}

Instructions for the agent:
- Follow the exact output structure:

**Comprehensive Causality Assessment for <ADVERSE EVENT> Associated with <DRUG>**

**FINAL PLAUSIBILITY RATING:** HIGH / MEDIUM / LOW

**Detailed Causality Analysis:**
1. Temporal Plausibility
2. Biological Plausibility
3. Dechallenge/Rechallenge
4. Objective Evidence
5. Confounding Factors
6. History of Similar Reactions
7. Synthesis and Final Assessment

- Explicitly include Dechallenge/Rechallenge results, literature evidence (PMIDs), and patient metadata.
- Base all reasoning strictly on the provided information.

""",
        agent=safety_assessor,
        expected_output="Comprehensive causality assessment structured as requested",
    )


# ---------------------------
# Auto-fill missing template variables in CrewAI task descriptions
# ---------------------------
_TEMPLATE_VAR_RE = re.compile(r"{([^{}]+)}")

def _find_template_vars(text: str) -> set[str]:
    return set(_TEMPLATE_VAR_RE.findall(text or ""))

def _ensure_required_inputs(tasks, inputs):
    required = set()
    for t in tasks:
        required |= _find_template_vars(getattr(t, "description", ""))

    for key in required:
        if key not in inputs:
            inputs[key] = "MISSING"

    return inputs


In [13]:
#@title 11. Pipeline for one case (software/agents/pipeline.py)
# ════════════════════════════════════════════════════════════════════════════
# Pipeline
# ════════════════════════════════════════════════════════════════════════════





async def run_case_report_plausibility_assessment(df_case, df_reactions, df_drugs, rag=None, max_pairs=0):
    """Agents 1 + 4 (CrewAI) and selective RAG for every drug-event pair of one case."""
    drug_event_pairs = _parse_drugs_and_events(df_drugs, df_reactions)
    if max_pairs:  # notebook: MAX_PAIRS
        drug_event_pairs = drug_event_pairs[:max_pairs]
        console.print(f"ℹ️ MAX_PAIRS={max_pairs}: processing {len(drug_event_pairs)} pair(s).", style="yellow")
    drug_causality_map = _extract_drug_causality_map(df_drugs)
    case_metadata = _extract_case_metadata(df_case)

    case_event_date = "MISSING"
    if not df_case.empty:
        case_event_date = _get_value_or_missing(df_case.iloc[0].get("Event Date", "MISSING"))
    case_id = str(df_case.iloc[0].get("Case ID", "")) if not df_case.empty else ""
    case_text = f"Case ID: {case_id or 'N/A'}"

    all_results, agent_logs = {}, []

    for i, (drug, event) in enumerate(drug_event_pairs, 1):
        console.print(f"\n--- Assessment {i}/{len(drug_event_pairs)}: {drug} vs {event} ---",
                      style="bold yellow on blue")

        temporal_causality_context = build_case_context(drug, case_metadata, drug_causality_map,
                                                        case_event_date)
        literature_evidence = search_pubmed(drug, event, max_results=3)
        _, literature_content = save_literature_to_file(drug, event, literature_evidence, "plausibility")

        lit_output = assess_output = ""
        pair_result = None
        if config.RUN_CREW:
            from crewai import Crew
            literature_analyst = build_literature_agent(drug, event)
            safety_assessor = build_safety_assessor(drug, event)
            task_lit = build_literature_task(literature_analyst, literature_content)
            task_assess = build_assessment_task(safety_assessor, drug, event, temporal_causality_context)
            crew = Crew(agents=[literature_analyst, safety_assessor],
                        tasks=[task_lit, task_assess], verbose=False)
            inputs = {
                "case_report_context": case_text,
                "drug_event_pair": f"{drug} vs {event}",
                "EDSS": case_metadata.get("edss", "MISSING"),
            }
            inputs = _ensure_required_inputs([task_lit, task_assess], inputs)
            pair_result = await crew.kickoff_async(inputs=inputs)
            lit_output = task_lit.output.raw if hasattr(task_lit.output, "raw") else str(task_lit.output)
            assess_output = task_assess.output.raw if hasattr(task_assess.output, "raw") else str(task_assess.output)

        # ── RAG: Q2/Q5/Q10 dual answers (no-RAG vs expert-informed) ──
        rag_entry = {}
        if rag is not None:
            rag_context = temporal_causality_context + "\n\n" + literature_content[:config.LITERATURE_CONTEXT_CHARS]
            try:
                rag_entry = rag.run_rag_for_pair(drug, event, rag_context, case_id)
            except Exception as err:
                console.print(f"⚠️  RAG step failed for {drug}/{event}: {err}", style="yellow")

        log = {"Drug": drug, "Adverse Event": event,
               "Literature Agent Summary": lit_output, "Safety Assessor Assessment": assess_output}
        for q in ("Q2", "Q5", "Q10"):
            log[f"{q}_no_rag"] = rag_entry.get(q, {}).get("version_no_rag", "")
            log[f"{q}_with_rag"] = rag_entry.get(q, {}).get("version_with_rag", "")
        agent_logs.append(log)
        all_results[f"{drug} vs {event}"] = pair_result
        console.print(f"✅ Assessment complete for {drug} vs {event}", style="bold green")

    final_report = "### Comprehensive Causality Assessment Report\n\n"
    final_report += "This report analyzes all drug-event pairs from the submitted case report.\n\n---\n"
    for pair, result in all_results.items():
        final_report += f"## Pair Assessment: {pair}\n**CrewAI Final Output:**\n```\n{result}\n```\n\n---\n"

    return final_report, pd.DataFrame(agent_logs)


async def assess_case_async(case_text: str, rag=None, deduplicate: bool = True, max_pairs: int = 0) -> dict:
    """Run the full MARCA assessment for one case (use `await` inside Jupyter/Colab)."""
    config.ensure_dirs()
    if rag is not None:
        rag.reset()
    text = deduplicate_drugs(case_text) if deduplicate else case_text
    df_case, df_reactions, df_drugs, df_comprehensive = process_case(text, display=True)

    drugs = df_drugs["Drug Name"].tolist() if "Drug Name" in df_drugs else []
    events = df_reactions["Adverse Reaction"].tolist() if "Adverse Reaction" in df_reactions else []

    q1 = assess_listedness(drugs, events)                 # Agent 2
    q10 = classify_aes(events)                            # Agent 3
    final_report, df_agent_logs = await run_case_report_plausibility_assessment(
        df_case, df_reactions, df_drugs, rag, max_pairs=max_pairs)   # Agents 1 + 4, RAG

    return {
        "case_id": str(df_case.iloc[0].get("Case ID", "")) if not df_case.empty else "",
        "df_case": df_case, "df_reactions": df_reactions, "df_drugs": df_drugs,
        "df_comprehensive": df_comprehensive,
        "q1_listedness": pd.DataFrame(q1), "q10_objective_evidence": pd.DataFrame(q10),
        "agent_logs": df_agent_logs, "final_report": final_report,
        "rag_results": list(rag.results) if rag is not None else [],
    }


def assess_case(case_text: str, rag=None, deduplicate: bool = True) -> dict:
    """Synchronous wrapper for scripts. In Jupyter/Colab use `await assess_case_async(...)`."""
    return asyncio.run(assess_case_async(case_text, rag, deduplicate))


def save_assessment_excel(result: dict, out_dir=None, extra_sheets=None) -> Path:
    """Write output/results/FDA_Case_<id>_Assessment.xlsx (one sheet per output, as in the notebook)."""
    out_dir = Path(out_dir or config.RESULTS_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"FDA_Case_{result['case_id'] or 'Unknown'}_Assessment.xlsx"
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for key, sheet in (("df_case", "Case_Info"), ("df_drugs", "Drug_Info"),
                           ("df_reactions", "Adverse_Reactions"), ("df_comprehensive", "Comprehensive_Data"),
                           ("q1_listedness", "Q1_Label_Listedness"),
                           ("q10_objective_evidence", "Q10_Sign_vs_Symptom"),
                           ("agent_logs", "Agent_Outputs_Split")):
            df = result.get(key)
            if isinstance(df, pd.DataFrame) and not df.empty:
                df.to_excel(writer, sheet_name=sheet, index=False)
        pd.DataFrame({"Comprehensive Plausibility Report": [result["final_report"]]}).to_excel(
            writer, sheet_name="Plausibility_Report_Text", index=False)
        rag_rows = []
        for entry in result.get("rag_results", []):
            for q in ("Q2", "Q5", "Q10"):
                qd = entry.get(q, {})
                refs = qd.get("retrieved_refs", [])
                rag_rows.append({
                    "Drug": entry["drug"], "Adverse_Event": entry["pt"], "Question": q,
                    "Answer_No_RAG": qd.get("version_no_rag", ""),
                    "Answer_With_RAG": qd.get("version_with_rag", ""),
                    "Expert_Refs": "; ".join(
                        f"{r['drug']}/{r['pt']} (expert {q}={r.get(f'{q}_h_score', '?')}, sim={r.get('_sim', 0):.0%})"
                        for r in refs) or "none",
                })
        pd.DataFrame(rag_rows or [{"Message": "RAG module not run or no results."}]).to_excel(
            writer, sheet_name="RAG_Q2_Q5_Q10", index=False)
        for sheet, df in (extra_sheets or {}).items():  # notebook: extra sheets
            df.to_excel(writer, sheet_name=sheet[:31], index=False)
    console.print(f"✅ Saved {path}", style="green")
    return path


In [14]:
#@title 12. Selective RAG for Q2, Q5, Q10 (software/rag/__init__.py, unchanged)
"""Selective RAG for Naranjo Q2, Q5 and Q10.

Contents: drug-class resolution (openFDA -> RxNorm -> DeepSeek, cached), expert knowledge
base loading and TF-IDF retrieval indices, prompt text, and the SelectiveRAG class.
"""
import os
import re as _re
import json
import time
from collections import defaultdict

import numpy as np
import openpyxl
import requests as _req
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



# ════════════════════════════════════════════════════════════════════════════
# Drug-class resolution
# ════════════════════════════════════════════════════════════════════════════


def normalise_class(raw: str) -> str:
    s = _re.sub(r"\s*\[EPC\]", "", raw).strip()
    s = _re.sub(r"[^\w\s]", "", s)
    s = _re.sub(r"\s+", "_", s)
    return s.upper() or "UNKNOWN"


def openfda_epc(drug_name: str):
    primary = drug_name.split("\\")[0].strip().lower()
    names   = ([drug_name.lower()] if drug_name.lower() != primary else []) + [primary]
    for name in names:
        for field in ("openfda.brand_name", "openfda.substance_name",
                      "openfda.generic_name"):
            try:
                r = _req.get("https://api.fda.gov/drug/label.json",
                             params={"search": f'{field}:"{name}"', "limit": 1},
                             timeout=10)
                if r.status_code != 200:
                    continue
                epc = r.json().get("results", [{}])[0].get("openfda", {}) \
                               .get("pharm_class_epc", [])
                if epc:
                    return normalise_class(epc[0])
            except Exception:
                pass
    return None




def rxnorm_class(drug_name: str):
    primary = drug_name.split("\\")[0].strip().lower()
    try:
        r = _req.get("https://rxnav.nlm.nih.gov/REST/rxcui.json",
                     params={"name": primary, "search": 2}, timeout=10)
        if r.status_code != 200:
            return None
        cuis = r.json().get("idGroup", {}).get("rxnormId", [])
        if not cuis:
            return None
        for source in ("ATC", "VA", "MESH"):
            r2 = _req.get(
                "https://rxnav.nlm.nih.gov/REST/rxclass/class/byRxcui.json",
                params={"rxcui": cuis[0], "relaSource": source}, timeout=10)
            items = (r2.json().get("rxclassDrugInfoList", {})
                              .get("rxclassDrugInfo", [])
                     if r2.status_code == 200 else [])
            if items:
                best = max(items,
                           key=lambda x: len(x["rxclassMinConceptItem"]["classId"]))
                return normalise_class(best["rxclassMinConceptItem"]["className"])
    except Exception:
        pass
    return None

def deepseek_batch(drugs: list) -> dict:
    if not drugs:
        return {}
    prompt = (
        "You are a senior pharmacologist. For each drug below return its "
        "pharmacological class as a concise uppercase label with underscores "
        "(e.g. SULFONYLUREA, BRAF_INHIBITOR, ANTI_TNF_BIOLOGIC, INSULIN, SSRI).\n"
        "For combination products use the primary active component class.\n"
        "Return ONLY a JSON object mapping each exact drug name to its class.\n\n"
        "Drugs:\n" + "\n".join(f"- {d}" for d in drugs)
    )
    try:
        parsed = call_deepseek_json(prompt, system="Return only valid JSON, no other text.", timeout=90)
        return {k: normalise_class(str(v)) for k, v in parsed.items() if k in drugs}
    except Exception as e:
        print(f"    ⚠️  DeepSeek batch error: {e}")
        return {}



BATCH_SIZE = 30
FDA_PAUSE = 0.25
RXNORM_PAUSE = 0.12


def build_class_map(unique_drugs: list, cache_path=None) -> dict:
    """
    Resolve drug → class for all unique_drugs.
    Loads from cache if available; only queries APIs for new drugs.
    """
    cache_path = str(cache_path or config.DRUG_CLASS_CACHE_PATH)
    cache: dict = {}
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            cache = json.load(f)
        to_resolve = [d for d in unique_drugs if d.upper() not in cache]
        if not to_resolve:
            print(f"[DrugClass] ✓ All {len(unique_drugs)} drugs in cache.")
            return cache
        print(f"[DrugClass] Cache hit for "
              f"{len(unique_drugs)-len(to_resolve)}/{len(unique_drugs)} drugs. "
              f"Resolving {len(to_resolve)} new ones…")
    else:
        to_resolve = unique_drugs
        print(f"[DrugClass] No cache. Resolving {len(to_resolve)} drugs…")

    unresolved = []
    hits1 = 0
    print("  [1/3] OpenFDA…")
    for i, drug in enumerate(to_resolve, 1):
        cls = openfda_epc(drug)
        if cls:
            cache[drug.upper()] = cls
            hits1 += 1
        else:
            unresolved.append(drug)
        if i % 20 == 0 or i == len(to_resolve):
            print(f"    {i}/{len(to_resolve)}  ({hits1} resolved so far)")
        time.sleep(FDA_PAUSE)
    print(f"  OpenFDA: {hits1} resolved, {len(unresolved)} remain.")

    still = []
    hits2 = 0
    print("  [2/3] RxNorm…")
    for drug in unresolved:
        cls = rxnorm_class(drug)
        if cls:
            cache[drug.upper()] = cls
            hits2 += 1
        else:
            still.append(drug)
        time.sleep(RXNORM_PAUSE)
    print(f"  RxNorm: {hits2} resolved, {len(still)} remain.")

    if still:
        hits3 = 0
        print(f"  [3/3] DeepSeek batch ({len(still)} drugs)…")
        for start in range(0, len(still), BATCH_SIZE):
            batch = still[start: start + BATCH_SIZE]
            result = deepseek_batch(batch)
            for drug, cls in result.items():
                cache[drug.upper()] = cls
                hits3 += 1
            for drug in batch:
                if drug.upper() not in cache:
                    cache[drug.upper()] = "UNKNOWN"
        print(f"  DeepSeek: {hits3} resolved.")

    with open(cache_path, "w") as f:
        json.dump(cache, f, indent=2, sort_keys=True)
    print(f"[DrugClass] Cache saved → {cache_path}")
    return cache


# ════════════════════════════════════════════════════════════════════════════
# Expert knowledge base and retrieval indices
# ════════════════════════════════════════════════════════════════════════════


# ── KB column positions (0-based) ────────────────────────────────────────────
_COL_CASE_ID = 1
_COL_DRUG    = 3
_COL_PT      = 2
_COL_NARR    = 47
_COL_META    = 51
_RAG_Q = {
    "Q2":  {"h_score": 10, "h_reason": 11},
    "Q5":  {"h_score": 22, "h_reason": 23},
    "Q10": {"h_score": 42, "h_reason": 43},
}

# ── Missing-date / missing-confounder detectors ──────────────────────────────
_DATE_FIELDS = ["Therapy Start Date", "Therapy End Date", "Event Date",
                "therapy start", "event date"]
_NA_TOKENS   = {"na", "nan", "missing", "unknown", "unk", "n/a", ""}

def has_any_date(ctx: str) -> bool:
    for field in _DATE_FIELDS:
        m = _re.search(rf"{_re.escape(field)}\s*[:\-]\s*([^\n;│]+)",
                       ctx, _re.IGNORECASE)
        if m:
            val = m.group(1).strip().lower()
            if val not in _NA_TOKENS and len(val) >= 4:
                return True
    return False

def has_any_confounder_info(ctx: str) -> bool:
    signals = ["concurrent", "concomitant", "co-medication", "co-morbid",
               "indication for use", "underlying condition", "medical history",
               "confound"]
    return any(s in ctx.lower() for s in signals)


def _named_layout(header_row) -> dict | None:
    """If the first row holds column names (MARCA_vs_expert_comparison layout), map them."""
    names = [str(v).strip() if v is not None else "" for v in header_row]
    if "q2_score_human" not in names:
        return None
    idx = {n: i for i, n in enumerate(names)}
    rag_q = {q: {"h_score": idx[f"{q.lower()}_score_human"], "h_reason": idx[f"{q.lower()}_reasoning_human"]}
             for q in ("Q2", "Q5", "Q10")}
    return {"case_id": idx["case_id"], "pt": idx["pt"], "drug": idx["drug_name"],
            "narr": idx["case_narrative"], "meta": idx["questions2_9_ai_answer"],
            "rag_q": rag_q, "first_row": 2}


def kb_layout(path: str) -> dict:
    """Column positions of the KB workbook.

    Two layouts are supported:
      * named columns in row 1 (e.g. MARCA_vs_expert_comparison.xlsx: q2_score_human, …);
      * the original study workbook without headers in row 1, data from row 3
        (fixed positions defined at the top of this module).
    """
    wb = openpyxl.load_workbook(path, read_only=True)
    header = next(wb.active.iter_rows(min_row=1, max_row=1, values_only=True))
    wb.close()
    return _named_layout(header) or {
        "case_id": _COL_CASE_ID, "pt": _COL_PT, "drug": _COL_DRUG,
        "narr": _COL_NARR, "meta": _COL_META, "rag_q": _RAG_Q, "first_row": 3}


def load_expert_kb(path: str, class_map: dict) -> list:
    if not os.path.exists(path):
        print(f"⚠️  KB not found at {path}")
        return []
    L = kb_layout(path)
    wb = openpyxl.load_workbook(path, read_only=True)
    ws = wb.active
    records = []
    for row in ws.iter_rows(min_row=L["first_row"], values_only=True):
        if not any(row):
            continue
        drug = str(row[L["drug"]] or "").strip()
        pt   = str(row[L["pt"]]   or "").strip()
        if not drug or not pt:
            continue
        narr = str(row[L["narr"]] or "")
        meta = str(row[L["meta"]] or "")
        rec = {
            "case_id":        str(row[L["case_id"]] or "").strip(),
            "drug":           drug,
            "pt":             pt,
            "drug_class":     class_map.get(drug.upper(), "UNKNOWN"),
            "narrative":      narr,
            "metadata":       meta,
            "no_dates":       not has_any_date(narr),
            "no_confounders": not has_any_confounder_info(narr + " " + meta),
        }
        for qname, cols in L["rag_q"].items():
            rec[f"{qname}_h_score"]  = row[cols["h_score"]]
            rec[f"{qname}_h_reason"] = str(row[cols["h_reason"]] or "")
        records.append(rec)
    wb.close()
    print(f"[KB] Loaded {len(records)} records, "
          f"{len({r['drug'] for r in records})} unique drugs.")
    return records


class ExpertVectorDB:
    def __init__(self, records: list):
        self.records = records
        if not records:
            return

        # exact case_id lookup
        self._by_id = {r["case_id"]: r for r in records}

        # ── flat indices (v2 style, global over all records) ──────────────
        ae_corpus  = [r["pt"].lower() for r in records]
        ctx_corpus = [
            f"{r['drug'].lower()} {r['pt'].lower()} {r['narrative'][:400].lower()}"
            for r in records
        ]
        self._ae_vec  = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
        self._ae_mat  = self._ae_vec.fit_transform(ae_corpus)
        self._ctx_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_features=5000)
        self._ctx_mat = self._ctx_vec.fit_transform(ctx_corpus)

        # ── per-drug flat index (Tier 1) ──────────────────────────────────
        # corpus: pt text only; searched by pt query
        drug_buckets  = defaultdict(list)
        class_buckets = defaultdict(list)
        for r in records:
            drug_buckets[r["drug"].upper()].append(r["pt"].lower())
            if r["drug_class"] != "UNKNOWN":
                class_buckets[r["drug_class"]].append(r["pt"].lower())

        self._drug_recs  = defaultdict(list)
        self._class_recs = defaultdict(list)
        for r in records:
            self._drug_recs[r["drug"].upper()].append(r)
            if r["drug_class"] != "UNKNOWN":
                self._class_recs[r["drug_class"]].append(r)

        self._drug_idx  = {}   # drug_upper  → (vec, mat)
        self._class_idx = {}   # class_upper → (vec, mat)

        for drug_key, pts in drug_buckets.items():
            vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
            self._drug_idx[drug_key] = (vec, vec.fit_transform(pts))

        for cls_key, pts in class_buckets.items():
            vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
            self._class_idx[cls_key] = (vec, vec.fit_transform(pts))

        # ── missing-date sub-pool (v2 style) ─────────────────────────────
        nd = [r for r in records if r["no_dates"]]
        self._no_date_records = nd
        if nd:
            nd_corpus = [
                f"{r['drug'].lower()} {r['pt'].lower()}" for r in nd
            ]
            self._nd_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
            self._nd_mat = self._nd_vec.fit_transform(nd_corpus)
        else:
            self._nd_vec = self._nd_mat = None

        n_d = len(self._drug_idx)
        n_c = len(self._class_idx)
        print(f"[VectorDB] Built — {n_d} drug buckets, {n_c} class buckets, "
              f"{len(nd)} no-date records.")

    # ── shared scorer (same logic as v2 _query) ───────────────────────────
    def _query(self, mat, vec, pool: list, text: str,
               top_k: int, exclude_id: str) -> list:
        q    = vec.transform([text.lower()])
        sims = cosine_similarity(q, mat).flatten()
        order = np.argsort(sims)[::-1]
        out   = []
        for i in order:
            r = pool[i]
            if exclude_id and r["case_id"] == exclude_id:
                continue
            if sims[i] < 0.01:
                break
            out.append({**r, "_sim": float(sims[i])})
            if len(out) >= top_k:
                break
        return out

    # ── exact lookup ──────────────────────────────────────────────────────
    def lookup_exact(self, case_id) -> dict | None:
        return self._by_id.get(str(case_id).strip())

    # ── two-tier retrieval ────────────────────────────────────────────────
    def query_two_tier(self, drug: str, pt: str,
                       top_k: int = 3, exclude_id: str = "") -> list:
        """
        Tier 1: records with the SAME drug, ranked by AE similarity.
        Tier 2: records with the SAME drug class, ranked by AE similarity.
        Falls back to global context search if both tiers are empty.
        """
        drug_key  = drug.strip().upper()
        drug_class = "UNKNOWN"
        # get class from any record that matches this drug
        for r in self.records:
            if r["drug"].upper() == drug_key:
                drug_class = r["drug_class"]
                break

        results  = []
        seen_ids = set()

        # Tier 1 — same drug, search by PT
        if drug_key in self._drug_idx:
            vec, mat = self._drug_idx[drug_key]
            pool = self._drug_recs[drug_key]
            for r in self._query(mat, vec, pool, pt, top_k, exclude_id):
                if r["case_id"] not in seen_ids:
                    results.append({**r, "_sim_tier": "exact_drug"})
                    seen_ids.add(r["case_id"])

        # Tier 2 — same class, search by PT (fill remaining slots)
        if len(results) < top_k and drug_class not in ("UNKNOWN", "", None):
            if drug_class in self._class_idx:
                vec, mat = self._class_idx[drug_class]
                pool = self._class_recs[drug_class]
                for r in self._query(mat, vec, pool, pt, top_k * 3, exclude_id):
                    if r["case_id"] not in seen_ids:
                        results.append({**r, "_sim_tier": "same_class"})
                        seen_ids.add(r["case_id"])
                    if len(results) >= top_k:
                        break

        # Fallback — global context search (v2 behaviour)
        if not results:
            for r in self._query(self._ctx_mat, self._ctx_vec, self.records,
                                 f"{drug} {pt}", top_k, exclude_id):
                if r["case_id"] not in seen_ids:
                    results.append({**r, "_sim_tier": "global"})
                    seen_ids.add(r["case_id"])

        return results[:top_k]

    # ── AE-only search (for Q10) ──────────────────────────────────────────
    def query_ae(self, pt: str, top_k: int = 2, exclude_id: str = "") -> list:
        return [
            {**r, "_sim_tier": "ae_match"}
            for r in self._query(self._ae_mat, self._ae_vec,
                                 self.records, pt, top_k, exclude_id)
        ]

    # ── missing-date pool search ──────────────────────────────────────────
    def query_missing_dates(self, drug: str, pt: str,
                            top_k: int = 3, exclude_id: str = "") -> list:
        if self._nd_mat is None or not self._no_date_records:
            return []
        return [
            {**r, "_sim_tier": "no_date_pool"}
            for r in self._query(self._nd_mat, self._nd_vec,
                                 self._no_date_records,
                                 f"{drug} {pt}", top_k, exclude_id)
        ]


# ════════════════════════════════════════════════════════════════════════════
# Prompts
# ════════════════════════════════════════════════════════════════════════════
_EXPERT_Q2_NO_DATES = (
    "the case does not provide any information on therapy start and end date "
    "or on the event date, thus temporal and biological plausibility cannot be established."
)
_EXPERT_Q5_NO_INFO = (
    "No information is provided about confounding factors or concurrent risk factors "
    "in the report. Confounding cannot be assessed."
)

Q_DEF = {
    "Q2": (
        "Naranjo Q2 – Did the adverse event appear after the suspected drug was administered?\n"
        "Allowed scores: +2=Yes (documented event after exposure); "
        "-1=No (event clearly preceded exposure); 0=Do not know (timing indeterminate).\n"
        "If timing cannot be established from case data, use 0; do not infer dates."
    ),
    "Q5": (
        "Naranjo Q5 – Are there alternative causes that could independently explain the reaction?\n"
        "Allowed scores: -1=Yes (credible alternative cause); "
        "+2=No (credible alternative causes excluded using case-specific evidence); "
        "0=Do not know (insufficient clinical information).\n"
        "Absence of documented confounders is not proof that alternatives were excluded."
    ),
    "Q10": (
        "Naranjo Q10 – Was the adverse event confirmed by objective evidence?\n"
        "Allowed scores: +1=Yes (documented patient-specific objective confirmation); "
        "0=No or Do not know (not confirmed or unavailable).\n"
        "A MedDRA term that denotes a measurable sign, by itself, is not patient-specific confirmation."
    ),
}


def build_ref_block(refs, qname):
    lines = []
    for i, r in enumerate(refs, 1):
        tier = r.get("_sim_tier", "")
        tier_label = (
            "same drug"  if tier == "exact_drug"  else
            f"same class · {r.get('drug_class','?')}" if tier == "same_class" else
            "no-date pool" if tier == "no_date_pool" else
            "global match"
        )
        lines.append(
            f"[Ref {i} | {tier_label} | sim={r.get('_sim', 0):.0%} | "
            f"case_id={r['case_id']} | {r['drug']} / {r['pt']}]\n"
            f"  Expert {qname} score  : {r.get(f'{qname}_h_score', '?')}\n"
            f"  Expert {qname} reason : {r.get(f'{qname}_h_reason', '') or '(none)'}"
        )
    return "\n\n".join(lines)


# ════════════════════════════════════════════════════════════════════════════
# Selective RAG
# ════════════════════════════════════════════════════════════════════════════




def _call_llm(prompt_user, system_msg=None):
    """DeepSeek call that returns an error string instead of raising (as in the notebook)."""
    try:
        if system_msg:
            return call_deepseek(prompt_user, system=system_msg)
        return call_deepseek(prompt_user)
    except Exception as e:
        return f"[LLM error: {e}]"


def dual_answer(qname, drug, pt, context, db, case_id=""):
    case_block   = f"Drug: {drug}\nAdverse Event: {pt}\nContext:\n{context[:config.CASE_CONTEXT_CHARS]}"
    no_dates     = not has_any_date(context)
    no_conf_info = not has_any_confounder_info(context)
    expert_exact = db.lookup_exact(case_id)

    # Retrieval
    if qname == "Q10":
        similar = db.query_ae(pt, top_k=2, exclude_id=case_id)
    elif qname == "Q2" and no_dates:
        similar = db.query_missing_dates(drug, pt, top_k=3, exclude_id=case_id)
    else:
        similar = db.query_two_tier(drug, pt, top_k=3, exclude_id=case_id)
        if qname == "Q5" and no_conf_info:
            # prefer refs that also had no confounder info
            pref = [r for r in similar if r.get("no_confounders")]
            rest = [r for r in similar if not r.get("no_confounders")]
            similar = (pref + rest)[:3]

    # No-RAG
    ans_no_rag = _call_llm(
        f"Answer the following pharmacovigilance question.\n\n"
        f"Question:\n{Q_DEF[qname]}\n\n"
        f"Case data:\n{case_block}\n\n"
        f"Reply as:\nScore: <number>\nReasoning: <justification>"
    )

    # With-RAG
    rag_parts = []
    if qname == "Q2" and no_dates:
        rag_parts.append(
            "--- MANDATORY EXPERT RULE: MISSING DATES ---\n"
            "Both therapy start date and event date are NA/missing.\n"
            "Expert annotators consistently score Q2 = 0 and state:\n"
            f'  "{_EXPERT_Q2_NO_DATES}"\n'
            "You MUST apply this rule: score Q2 = 0."
        )
    if qname == "Q5" and no_conf_info:
        rag_parts.append(
            "--- MANDATORY EXPERT RULE: NO CONFOUNDER INFO ---\n"
            "No concurrent medications or comorbidities info available.\n"
            "Expert annotators score Q5 = 0 in this situation.\n"
            "You MUST apply this rule: score Q5 = 0."
        )
    ref_block = build_ref_block(similar, qname)
    if ref_block:
        rag_parts.append(
            f"--- Expert Reference Cases (annotated KB) ---\n{ref_block}\n"
            "--- End References ---\n"
            "Use these to calibrate your answer where relevant."
        )

    if rag_parts:
        ans_with_rag = _call_llm(
            f"Answer the following pharmacovigilance question.\n\n"
            f"Question:\n{Q_DEF[qname]}\n\n"
            f"Case data:\n{case_block}\n\n"
            + "\n\n".join(rag_parts)
            + "\n\nReply as:\nScore: <number>\nReasoning: <justification>"
        )
    else:
        ans_with_rag = ans_no_rag + "\n[No expert context available]"

    return ans_no_rag, ans_with_rag, similar, expert_exact


class SelectiveRAG:
    """Holds the expert KB and its retrieval indices; collects results per case."""

    def __init__(self, records: list):
        self.records = records
        self.db = ExpertVectorDB(records) if records else None
        self.results = []          # one entry per drug-event pair (was `rag_results`)

    @classmethod
    def from_workbook(cls, path=None, cache_path=None):
        """Load the KB workbook, resolve drug classes (cached) and build the indices."""
        path = str(path or config.KB_PATH)
        if not os.path.exists(path):
            print(f"\n❌  KB not found: {path}")
            return cls([])
        layout = kb_layout(path)
        wb = openpyxl.load_workbook(path, read_only=True)
        raw_drugs = [str(row[layout["drug"]]).strip()
                     for row in wb.active.iter_rows(min_row=layout["first_row"], values_only=True)
                     if row[layout["drug"]]]
        wb.close()
        class_map = build_class_map(sorted(set(raw_drugs)), cache_path=cache_path)
        records = load_expert_kb(path, class_map)
        rag = cls(records)
        if records:
            print(f"\n✅ RAG module ready. {len(records)} records | "
                  f"{len({r['drug'] for r in records})} drugs | "
                  f"{len({r.get('drug_class', 'UNKNOWN') for r in records})} classes")
        return rag

    def reset(self):
        """Clear results before assessing a new case."""
        self.results = []

    def run_rag_for_pair(self, drug: str, pt: str, context_text: str, case_id: str = "") -> dict:
        """Called once per drug-AE pair from the main pipeline."""
        if not self.records or self.db is None:
            print("⚠️  run_rag_for_pair: KB not loaded — skipping.")
            return {"drug": drug, "pt": pt, "case_id": case_id}

        entry = {"drug": drug, "pt": pt, "case_id": case_id}
        for qname in ["Q2", "Q5", "Q10"]:
            no_rag, with_rag, refs, exact = dual_answer(qname, drug, pt, context_text, self.db, case_id)
            entry[qname] = {
                "version_no_rag": no_rag,
                "version_with_rag": with_rag,
                "retrieved_refs": refs,
                "expert_exact": {
                    "case_id": exact["case_id"],
                    "h_score": exact.get(f"{qname}_h_score"),
                    "h_reason": exact.get(f"{qname}_h_reason", ""),
                } if exact else None,
            }
        self.results.append(entry)
        return entry


In [15]:
#@title 13. Score parsing and Naranjo categories (software/evaluation)
VALID_SCORES = {
    "Q1": {0, 1}, "Q2": {-1, 0, 2}, "Q3": {0, 1},
    "Q4": {-1, 0, 2}, "Q5": {-1, 0, 2}, "Q6": {-1, 0, 1},
    "Q7": {0, 1}, "Q8": {0, 1}, "Q9": {0, 1}, "Q10": {0, 1},
}


def parse_score(text, question=None):
    """Extract a validated Naranjo score from an answer such as 'Score: -1 ...'."""
    if not text:
        return None
    m = re.search(r"Score\s*[:\-]\s*([+-]?\d+(?:\.\d+)?)", str(text), re.IGNORECASE)
    if not m:
        return None
    value = float(m.group(1))
    if question is not None and value not in VALID_SCORES[question]:
        raise ValueError(f"Invalid Naranjo {question} score from model: {value}")
    return value


def naranjo_category(total):
    if pd.isna(total):
        return np.nan
    if total >= 9:
        return "Definite"
    if total >= 5:
        return "Probable"
    if total >= 1:
        return "Possible"
    return "Doubtful"


## 3. Data and the case

In [16]:
#@title 14. Expert knowledge base (expert_kb.xlsx) and Agent 2 label files
#@markdown The knowledge base is downloaded from the public repository. If that fails, you are asked to upload `expert_kb.xlsx`.


def download(url: str, dest: Path) -> bool:
    try:
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(r.content)
        return True
    except Exception as e:
        print(f"Download failed ({url}): {e}")
        return False


if not config.KB_PATH.exists():
    if not download(f"{GITHUB_RAW}/data/knowledge_base/expert_kb.xlsx", config.KB_PATH):
        from google.colab import files
        print("Upload expert_kb.xlsx")
        up = files.upload()
        name = next(iter(up))
        config.KB_PATH.write_bytes(up[name])

kb_df = pd.read_excel(config.KB_PATH)
print(f"Knowledge base: {len(kb_df)} DECs from {kb_df['case_id'].nunique()} ICSRs, "
      f"{kb_df['drug_name'].astype(str).str.strip().nunique()} drugs")

# Agent 2 needs one adverse-event list per drug. The original label files were not kept in the
# repository, so they are rebuilt from the study's recorded Agent 2 output ("Listed in label" events).
# These files only contain label events that occurred in the study DECs; for a drug that is not in the
# knowledge base, build its file from the FDA label PDF in the next cell, otherwise Agent 2 skips it.
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()) as _log:
    labels_from_study_workbook()
print(_log.getvalue().strip().splitlines()[-1])


Knowledge base: 1562 DECs from 215 ICSRs, 195 drugs
✅ 195 label files, 128 listed events, from /content/marca/data/knowledge_base/expert_kb.xlsx


In [19]:
#@title 15. (Optional) Build an Agent 2 label file from an FDA label PDF
#@markdown Only needed for a drug that is not in the knowledge base. Paste the Drugs@FDA label PDF URL and the drug name exactly as it appears in the case's `Drug Name` field.
LABEL_PDF_URL = "https://www.accessdata.fda.gov/drugsatfda_docs/label/2011/009218s107lbl.pdf"  #@param {type:"string"}
LABEL_DRUG_NAME = "COUMADIN"  #@param {type:"string"}

if LABEL_PDF_URL and LABEL_DRUG_NAME:
    build_label_csv(LABEL_PDF_URL, LABEL_DRUG_NAME)
else:
    print("Skipped (no URL given).")


📥 Downloading: https://www.accessdata.fda.gov/drugsatfda_docs/label/2011/009218s107lbl.pdf
✅ CSV file saved: /content/marca/data/knowledge_base/drug_labels/COUMADIN.csv


In [20]:
#@title 16. Build the selective RAG index
#@markdown With `RESOLVE_DRUG_CLASSES` on, every KB drug is mapped to a pharmacological class (openFDA → RxNorm → DeepSeek) and cached in `drug_class_map.json`; the study's own map was not kept, so DeepSeek-assigned classes may be worded differently. With it off, the same-class retrieval tier is skipped.
rag = None
if USE_RAG:
    # A cache written with RESOLVE_DRUG_CLASSES off holds only "UNKNOWN": drop it when resolving
    if RESOLVE_DRUG_CLASSES and config.DRUG_CLASS_CACHE_PATH.exists():
        cached = json.loads(config.DRUG_CLASS_CACHE_PATH.read_text())
        if cached and set(cached.values()) == {"UNKNOWN"}:
            config.DRUG_CLASS_CACHE_PATH.unlink()
    if not RESOLVE_DRUG_CLASSES and not config.DRUG_CLASS_CACHE_PATH.exists():
        drugs = sorted(set(kb_df["drug_name"].astype(str).str.strip()))
        config.DRUG_CLASS_CACHE_PATH.write_text(json.dumps({d.upper(): "UNKNOWN" for d in drugs}, indent=2))
        print("Drug classes not resolved: class-level retrieval tier disabled.")
    rag = SelectiveRAG.from_workbook()
else:
    print("RAG disabled in settings.")


[DrugClass] No cache. Resolving 195 drugs…
  [1/3] OpenFDA…
    20/195  (8 resolved so far)
    40/195  (17 resolved so far)
    60/195  (27 resolved so far)
    80/195  (38 resolved so far)
    100/195  (53 resolved so far)
    120/195  (66 resolved so far)
    140/195  (76 resolved so far)
    160/195  (91 resolved so far)
    180/195  (101 resolved so far)
    195/195  (109 resolved so far)
  OpenFDA: 109 resolved, 86 remain.
  [2/3] RxNorm…
  RxNorm: 83 resolved, 3 remain.
  [3/3] DeepSeek batch (3 drugs)…
  DeepSeek: 3 resolved.
[DrugClass] Cache saved → /content/marca/data/knowledge_base/drug_class_map.json
[KB] Loaded 1562 records, 195 unique drugs.
[VectorDB] Built — 195 drug buckets, 134 class buckets, 515 no-date records.

✅ RAG module ready. 1562 records | 195 drugs | 134 classes


In [21]:
#@title 17. The case to assess
#@markdown Leave `CASE_SOURCE` on **example** to use the repository's example case (FAERS case 229933175, icatibant, 10 events), choose **upload** to upload a `.txt` file in the same `Field: value` format, or **paste** to use the text in `PASTED_CASE` below.
CASE_SOURCE = "paste"  #@param ["example", "upload", "paste"]

PASTED_CASE = """
Case ID: 229933175
 Case Version: 5
 FDA Code: F
 Event Date: 20230314
 Manufacture Date: 20250214
 Initial FDA Date: 20230927
 FDA Date: 20250219
 Report Code: PER
 Authorization Number: NA
 Manufacturer Number: CO-TAKEDA-2023TUS029088
 Manufacturer Sender: TAKEDA
 Literature Reference: NA
 Age: 47
 Age Code: YR
 Age Group: NA
 Sex: F
 E Sub: Y
 Weight: 20
 Weight Code: KG
 Report Date: 20250219
 To Manufacturer: NA
 Occupation Code: NA
 Reporter Country: CO
 Occurrence Country: CO
 Adverse Reaction(s): Gastrointestinal Bleeding
 Case ID Number: 22993317
 Drug Sequence: 1
 Role Code: PS
 Drug Name: COUMADIN
 Product AI: WARFARIN
 Val VBM: 1
 Route: Oral
 Dose VBM: 1 dosage form
 Cumulative Dose (Char): NA
 Cumulative Dose (Unit): NA
 Dechal: NA
 Rechal: U
 Lot Number: NA
 Expiration Date: NA
 NDA Number: 22150
 Dose Amount: NA
 Dose Unit: NA
 Dose Form: Pill
 Dose Frequency: NA
 Indication for Use: Secondary prevention of MACE
 Therapy Start Date: NA
 Therapy End Date: NA
 Duration: NA
 Duration Code: NA
"""

if CASE_SOURCE == "example":
    example_path = config.RAW_DIR / "example_case.txt"
    if not example_path.exists():
        download(f"{GITHUB_RAW}/data/raw/example_case.txt", example_path)
    CASE_TEXT = example_path.read_text(encoding="utf-8")
elif CASE_SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    CASE_TEXT = up[next(iter(up))].decode("utf-8")
else:
    CASE_TEXT = PASTED_CASE

print(CASE_TEXT)



Case ID: 229933175
 Case Version: 5
 FDA Code: F
 Event Date: 20230314
 Manufacture Date: 20250214
 Initial FDA Date: 20230927
 FDA Date: 20250219
 Report Code: PER
 Authorization Number: NA
 Manufacturer Number: CO-TAKEDA-2023TUS029088
 Manufacturer Sender: TAKEDA
 Literature Reference: NA
 Age: 47
 Age Code: YR
 Age Group: NA
 Sex: F
 E Sub: Y
 Weight: 20
 Weight Code: KG
 Report Date: 20250219
 To Manufacturer: NA
 Occupation Code: NA
 Reporter Country: CO
 Occurrence Country: CO
 Adverse Reaction(s): Gastrointestinal Bleeding
 Case ID Number: 22993317
 Drug Sequence: 1 
 Role Code: PS 
 Drug Name: COUMADIN 
 Product AI: WARFARIN
 Val VBM: 1 
 Route: Oral
 Dose VBM: 1 dosage form 
 Cumulative Dose (Char): NA 
 Cumulative Dose (Unit): NA 
 Dechal: NA 
 Rechal: U 
 Lot Number: NA 
 Expiration Date: NA 
 NDA Number: 22150 
 Dose Amount: NA 
 Dose Unit: NA 
 Dose Form: Pill 
 Dose Frequency: NA 
 Indication for Use: Secondary prevention of MACE
 Therapy Start Date: NA 
 Therapy End Dat

## 4. Run

In [22]:
#@title 18. Run MARCA on the case
# Agent 2 (Q1) and Agent 3 (Q10) run on every drug–event pair / event;
# Agents 1 + 4 and RAG run on each pair (limited by MAX_PAIRS if set).
result = await assess_case_async(CASE_TEXT, rag, max_pairs=MAX_PAIRS)
print("\nDone. Case", result["case_id"], "—", len(result["agent_logs"]), "pair(s) assessed by Agents 1 + 4 / RAG.")


================================================================================

FDA CASE PARSED INTO DATAFRAMES

================================================================================

✅ Case ID: 229933175

✅ Drugs: 1

...

--- Assessment 1/1: COUMADIN vs Gastrointestinal Bleeding ---

🔍 Searching PubMed for: 'COUMADIN AND Gastrointestinal Bleeding adverse event'

# Agent: Expert Medical Searcher and Literature Summarizer
## Task: Summarize PubMed content:
--- Literature Search Results for: COUMADIN AND Gastrointestinal Bleeding ---
Date Generated: Tue Oct 20 13:39:06 2026
Articles Found: 3

ARTICLE 1
PMID: 42897786
Title: Nationwide Claims-Based Incidence and Associated Factors of Adverse Events after Gastric Endoscopic Submucosal Dissection.
Abstract:
Most data on adverse events (AEs) following gastric endoscopic submucosal dissection (ESD) originate from high-volume expert centers, limiting their generalizability to routine clinical practice. Population-level claims-based evidence on AEs after gastric ESD remains limited, although such data cannot fully account for lesion- and procedure-specific risk factors. We conducted a nationwide, retrospective cohort study using claims data from the Korean Health Insurance Review and Assessment Service. All patients who underwent ESD for gastric neoplastic lesions between 2013 and 2017 were included. Cl

✅ Assessment complete for COUMADIN vs Gastrointestinal Bleeding


Done. Case 229933175 — 1 pair(s) assessed by Agents 1 + 4 / RAG.


## 5. Results

In [23]:
#@title 19. Agent 2 (Q1) and Agent 3 (Q10) results
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 200)
display(Markdown("**Agent 2 — label listedness (Q1)**"))
display(result["q1_listedness"] if not result["q1_listedness"].empty
        else Markdown("_No label file for any drug of this case: Agent 2 skipped all pairs._"))
display(Markdown("**Agent 3 — sign vs symptom (Q10)**"))
display(result["q10_objective_evidence"])


**Agent 2 — label listedness (Q1)**

,drug,ae,status,explanation
0,COUMADIN,Gastrointestinal Bleeding,Not listed (Yes),"Coumadin (warfarin) is an anticoagulant that inhibits vitamin K-dependent clotting factors, increasing bleeding risk. Gastrointestinal bleeding is a well-documented and common adverse effect of wa..."


**Agent 3 — sign vs symptom (Q10)**

,ae,classification,value,rationale
0,Gastrointestinal Bleeding,Sign,1,"Gastrointestinal bleeding is an objective, observable, and measurable clinical finding (e.g., hematemesis, melena, hematochezia, or occult blood loss) that can be directly detected and quantified ..."


In [25]:
#@title 20. Agents 1 + 4 — literature summary and clinical causality report per DEC
logs = result["agent_logs"]
if logs.empty or not logs["Safety Assessor Assessment"].astype(str).str.strip().any():
    print("Agents 1 + 4 were not run (RUN_CREW is off) or returned nothing.")
else:
    for _, row in logs.iterrows():
        display(Markdown(f"---\n## {row['Drug']} — {row['Adverse Event']}"))
        display(Markdown("**Agent 1 — literature summary**\n\n" + str(row["Literature Agent Summary"])))
        display(Markdown("**Agent 4 — clinical causality report**\n\n" + str(row["Safety Assessor Assessment"])))


---
## COUMADIN — Gastrointestinal Bleeding

**Agent 1 — literature summary**

### **Analysis of PubMed Literature: Coumadin (Warfarin) and Gastrointestinal Bleeding**

This summary synthesizes the mechanism of action and clinical evidence regarding Coumadin (warfarin) and its association with gastrointestinal bleeding, based on the three provided PubMed articles.

---

### **Mechanism of Action (MOA) of Coumadin (Warfarin)**

The provided articles do not explicitly detail the biochemical mechanism of action of warfarin. However, they provide significant clinical context that alludes to its function and monitoring:

1.  **Classification:** Warfarin is identified as an oral anticoagulant (OAC) (PMID: 42699731).
2.  **Pharmacodynamic Monitoring:** The effect of warfarin is monitored via the International Normalized Ratio (INR). An "elevated international normalized ratio" is explicitly identified as a risk factor linked to an increased risk of bleeding from OACs in cancer patients (PMID: 42699731).
3.  **Clinical Management:** Warfarin therapy is contrasted with Direct Oral Anticoagulants (DOACs). DOACs are noted to be preferred in some settings because they "do not require routine blood monitoring," which implies that warfarin does require such monitoring to maintain a therapeutic and safe level of anticoagulation (PMID: 42696504).

**Established MOA (for context):** Warfarin is a vitamin K antagonist. It functions by inhibiting the enzyme vitamin K epoxide reductase, which is necessary for activating vitamin K in the body. This, in turn, interferes with the synthesis of vitamin K-dependent clotting factors—Factors II, VII, IX, and X—as well as the anticoagulant proteins C and S. The resulting depletion of functional clotting factors impairs the coagulation cascade, leading to an anticoagulant effect. This effect is measured by the Prothrombin Time (PT) and standardized as the INR.

---

### **Clinical Evidence from Provided Literature**

The articles provide direct and comparative evidence linking warfarin to an increased risk of gastrointestinal bleeding in specific clinical scenarios.

#### **1. Association with Post-Procedural Bleeding (PMID: 42897786)**

*   **Study Design:** A nationwide, retrospective cohort study in Korea using claims data from 83,871 gastric endoscopic submucosal dissection (ESD) procedures.
*   **Key Finding:** The study analyzed factors associated with adverse events following gastric ESD. Among the antithrombotic agents evaluated, **warfarin was specifically associated with an increased risk of post-procedural bleeding**.
*   **Clinical Context:** The overall incidence of claims-defined bleeding within 30 days of the procedure was significant at 16.6%. This finding positions warfarin as a notable risk factor in patients undergoing invasive gastrointestinal procedures.

#### **2. Comparative Bleeding Risk in Cancer Patients (PMID: 42699731)**

*   **Study Design:** An analysis of the FDA Adverse Event Reporting System (FAERS) database from 2014 to 2024, focusing on 29,897 adverse event reports of OACs in cancer patients.
*   **Key Finding:** When compared with the DOAC apixaban, **warfarin demonstrated an increased risk of bleeding events**.
*   **Site Specificity:** The study highlights that this increased bleeding risk associated with warfarin was **especially prominent in the gastrointestinal system**.
*   **Identified Risk Factors:** The study identified several factors that increase the risk of bleeding with OACs (including warfarin) in cancer patients:
    *   An elevated International Normalized Ratio (INR)
    *   Male sex
    *   Co-morbid cardiovascular diseases or cerebral infarction
    *   Renal insufficiency
    *   Presence of gastrointestinal cancer

#### **3. Clinical Practice and Management Context (PMID: 42696504)**

*   **Study Design:** A case report concerning life-threatening gastrointestinal bleeding associated with a DOAC (rivaroxaban).
*   **Key Finding (by comparison):** While the focus is on a DOAC, the report provides valuable context by stating that DOACs are "preferred over warfarin due to their rapid action and because they do not require routine blood monitoring." This underscores a fundamental clinical difference and highlights that the management of warfarin requires careful, routine monitoring (i.e., INR checks) to mitigate bleeding risk, a complexity not shared by DOACs. This reinforces the findings from the other articles that improper warfarin management (leading to an elevated INR) is a direct cause of bleeding complications.

**Agent 4 — clinical causality report**

**Comprehensive Causality Assessment for Gastrointestinal Bleeding Associated with COUMADIN**

**FINAL PLAUSIBILITY RATING:** HIGH

**Detailed Causality Analysis:**

**1. Temporal Plausibility**
The temporal relationship between the initiation of COUMADIN and the onset of Gastrointestinal Bleeding cannot be definitively established due to a MISSING Therapy Start date. The event is documented to have occurred on 20230314, during which the patient was actively receiving the drug ("Dose: 1 dosage form"). While a precise time-to-onset cannot be calculated, the event occurred while the patient was on therapy, which is consistent with a drug-induced adverse event. The lack of a start date is a significant data gap but does not rule out causality.

**2. Biological Plausibility**
The biological plausibility for COUMADIN (warfarin) causing Gastrointestinal Bleeding is extremely strong and well-established.
*   **Mechanism of Action:** As an oral anticoagulant (OAC) and vitamin K antagonist, COUMADIN's primary pharmacological effect is to inhibit the synthesis of vitamin K-dependent clotting factors (Factors II, VII, IX, X). This systemic impairment of the coagulation cascade inherently increases the risk of bleeding from any site, including the gastrointestinal tract.
*   **Literature Evidence:** The provided literature summary strongly supports this. It identifies an "elevated international normalized ratio" (INR), the measure of warfarin's effect, as a direct risk factor for bleeding (PMID: 42699731). Furthermore, a large FDA Adverse Event Reporting System (FAERS) analysis found that warfarin's increased bleeding risk was "especially prominent in the gastrointestinal system" when compared to other anticoagulants (PMID: 42699731).

**3. Dechallenge/Rechallenge**
Information regarding the effect of drug withdrawal (dechallenge) or re-administration (rechallenge) is unavailable.
*   **Dechallenge Result:** MISSING. It is unknown if the Gastrointestinal Bleeding resolved or improved after stopping COUMADIN.
*   **Rechallenge Result:** U (Unknown). The patient was not re-exposed to the drug, so the effect of a rechallenge cannot be assessed.
The absence of this information prevents its use in the causality assessment.

**4. Objective Evidence**
The reported event of 'Gastrointestinal Bleeding' is a clinically recognized adverse event. The association is strongly supported by objective, large-scale clinical data provided in the literature summary.
*   A nationwide retrospective cohort study (PMID: 42897786) specifically identified warfarin as being "associated with an increased risk of post-procedural bleeding" in the gastrointestinal tract.
*   A FAERS database analysis of over 29,000 adverse event reports (PMID: 42699731) demonstrated that warfarin carries an increased risk of bleeding events, particularly in the GI system.
This body of evidence provides a strong, objective basis for the causal link beyond the individual case report.

**5. Confounding Factors**
The potential for confounding factors appears to be low based on the provided metadata.
*   **Concurrent Medications:** The report explicitly states "N/A: single drug," which effectively rules out other medications as a cause for the bleeding event.
*   **Underlying Disease:** The patient's indication is "Secondary prevention of MACE." While cardiovascular disease can be a risk factor, the direct, potent anticoagulant mechanism of COUMADIN is the most probable precipitating cause of a major bleed. No other specific comorbidities (e.g., pre-existing GI lesions, renal insufficiency) were mentioned.

**6. History of Similar Reactions**
The patient's "History of Similar Reactions" is MISSING. Therefore, it is not possible to determine if the patient had a predisposition to bleeding events when exposed to anticoagulants in the past. This criterion neither supports nor refutes the assessment.

**7. Synthesis and Final Assessment**
This case presents a well-known and pharmacologically predictable adverse event. The primary strength of the causal association lies in the exceptionally strong biological plausibility; Gastrointestinal Bleeding is a direct consequence of COUMADIN's anticoagulant mechanism. This is heavily substantiated by multiple large-scale studies cited in the literature (PMID: 42699731, PMID: 42897786). A critical supporting factor is the absence of any concurrent medications, which eliminates potential drug-drug interactions or alternative drug causes.

The primary weaknesses are the significant gaps in case data, namely the MISSING therapy start date and the MISSING dechallenge result. These gaps prevent a full assessment of temporality and the effect of drug withdrawal. However, the strength of the pharmacological evidence and the lack of confounders are compelling enough to overcome these limitations. The event is not just plausible but an expected complication of COUMADIN therapy. Therefore, a **HIGH** plausibility rating is warranted.

In [26]:
#@title 21. Selective RAG — Q2, Q5, Q10 without and with expert references


def safe_parse(text, q):
    """Parsed score as int, None if the answer has no score, or 'invalid (...)' if outside Naranjo values."""
    try:
        v = parse_score(text, q)
        return None if v is None else int(v)
    except ValueError as e:
        return f"invalid ({e})"


rag_rows = []
for entry in result["rag_results"]:
    for q in ("Q2", "Q5", "Q10"):
        qd = entry.get(q, {})
        rag_rows.append({
            "Drug": entry["drug"], "Adverse event": entry["pt"], "Question": q,
            "Score no-RAG": safe_parse(qd.get("version_no_rag"), q),
            "Score with RAG": safe_parse(qd.get("version_with_rag"), q),
            "Retrieved expert references": "; ".join(
                f"{r['drug']}/{r['pt']} [{r.get('_sim_tier', '')}] expert {q}={r.get(f'{q}_h_score', '?')}, "
                f"sim={r.get('_sim', 0):.0%}" for r in qd.get("retrieved_refs", [])) or "none",
        })
rag_table = pd.DataFrame(rag_rows)
display(rag_table if not rag_table.empty else Markdown("_RAG was not run._"))

#@markdown Tick to print the full no-RAG and with-RAG answers (reasoning text).
SHOW_RAG_REASONING = False  #@param {type:"boolean"}
if SHOW_RAG_REASONING:
    for entry in result["rag_results"]:
        for q in ("Q2", "Q5", "Q10"):
            qd = entry.get(q, {})
            display(Markdown(f"**{entry['drug']} / {entry['pt']} — {q}**\n\n"
                             f"*No RAG:* {qd.get('version_no_rag', '')}\n\n"
                             f"*With RAG:* {qd.get('version_with_rag', '')}"))


,Drug,Adverse event,Question,Score no-RAG,Score with RAG,Retrieved expert references
0,COUMADIN,Gastrointestinal Bleeding,Q2,0,0,"DEFITELIO/Upper gastrointestinal haemorrhage [global] expert Q2=0, sim=16%; OLMESARTAN/Gastrointestinal disorder [global] expert Q2=0, sim=14%; DEXAMETHASONE/Gastrointestinal protozoal infection [..."
1,COUMADIN,Gastrointestinal Bleeding,Q5,0,0,"DEFITELIO/Upper gastrointestinal haemorrhage [global] expert Q5=-1, sim=16%; OLMESARTAN/Gastrointestinal disorder [global] expert Q5=-1, sim=14%; DEXAMETHASONE/Gastrointestinal protozoal infection..."
2,COUMADIN,Gastrointestinal Bleeding,Q10,0,0,"OLMESARTAN/Gastrointestinal disorder [ae_match] expert Q10=1, sim=39%; OLMESARTAN/Gastrointestinal toxicity [ae_match] expert Q10=1, sim=35%"


In [28]:
#@title 22. Provisional Naranjo score sheet (notebook addition)
#@markdown Where each score comes from:
#@markdown - **Q1**: Agent 2 status. "Listed in label" or "Not listed (Yes)" → +1, otherwise 0. This is the correspondence found in the study workbook (`q1_ai_answer` vs `q1_score_ai`); the paper does not state it.
#@markdown - **Q2, Q5**: the selective-RAG answer (with or without expert references, see `Q2_Q5_SOURCE`).
#@markdown - **Q10**: Agent 3 (Sign = 1, Symptom = 0), as in the study workbook. The RAG Q10 answer is shown next to it for comparison.
#@markdown - **Q3, Q4, Q6–Q9**: 0 until a reviewer scores them from Agent 4's report (manuscript Section 2.4.5). Enter reviewer scores in `MANUAL_SCORES`; they override any automatic value.
Q2_Q5_SOURCE = "no_rag"  #@param ["with_rag", "no_rag"]

MANUAL_SCORES = {
    # ("ICATIBANT", "Epistaxis"): {"Q3": 0, "Q4": 0},
}

QS = [f"Q{i}" for i in range(1, 11)]


def norm_drug(x):
    return re.sub(r"\s+", " ", str(x).replace("_x000D_", "")).strip().upper()


def norm_ae(x):
    return re.sub(r"\s+", " ", str(x).replace("_x000D_", "")).strip().lower()


def q1_from_status(status):
    s = str(status or "")
    if s == "Listed in label" or re.match(r"Not listed \(\s*Yes", s, re.I):
        return 1
    if s.startswith("Not listed"):
        return 0
    return None   # no label file or error: needs review


q1_map = {}
if not result["q1_listedness"].empty:
    for _, r in result["q1_listedness"].iterrows():
        q1_map[(norm_drug(r["drug"]), norm_ae(r["ae"]))] = r["status"]

events_in_order = unique_events(result["df_reactions"]["Adverse Reaction"].tolist())
q10_map = {}
for ae, r in zip(events_in_order, result["q10_objective_evidence"].to_dict("records")):
    v = r.get("value")
    q10_map[norm_ae(ae)] = int(v) if v in (0, 1, "0", "1") else None

rag_map = {(norm_drug(e["drug"]), norm_ae(e["pt"])): e for e in result["rag_results"]}
manual = {(norm_drug(d), norm_ae(a)): v for (d, a), v in MANUAL_SCORES.items()}

pairs = (result["agent_logs"][["Drug", "Adverse Event"]].itertuples(index=False)
         if not result["agent_logs"].empty else [])
rows = []
for drug, ae in pairs:
    key = (norm_drug(drug), norm_ae(ae))
    row = {"Drug": drug, "Adverse event": ae}
    status = q1_map.get((norm_drug(clean_name(drug)), norm_ae(clean_name(ae))))
    row["Q1"] = q1_from_status(status)
    entry = rag_map.get(key, {})
    field = "version_with_rag" if Q2_Q5_SOURCE == "with_rag" else "version_no_rag"
    for q in ("Q2", "Q5"):
        v = safe_parse(entry.get(q, {}).get(field), q)
        row[q] = v if isinstance(v, int) else None
    row["Q10"] = q10_map.get(norm_ae(ae))
    for q in ("Q3", "Q4", "Q6", "Q7", "Q8", "Q9"):
        row[q] = 0
    for q, v in manual.get(key, {}).items():
        row[q] = v
    missing = [q for q in QS if row[q] is None]
    row["Total"] = None if missing else sum(row[q] for q in QS)
    row["Category"] = naranjo_category(row["Total"]) if row["Total"] is not None else "incomplete"
    row["Needs review"] = ", ".join(missing + [q for q in ("Q3", "Q4", "Q6", "Q7", "Q8", "Q9")
                                               if q not in manual.get(key, {})])
    rag_q10 = safe_parse(entry.get("Q10", {}).get("version_with_rag"), "Q10")
    row["Q10 (RAG answer)"] = rag_q10
    row["Agent 2 status"] = status or "no label file"
    rows.append(row)

score_sheet = pd.DataFrame(rows, columns=["Drug", "Adverse event"] + QS +
                           ["Total", "Category", "Needs review", "Q10 (RAG answer)", "Agent 2 status"])
display(score_sheet)


,Drug,Adverse event,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,Total,Category,Needs review,Q10 (RAG answer),Agent 2 status
0,COUMADIN,Gastrointestinal Bleeding,1,0,0,0,0,0,0,0,0,1,2,Possible,"Q3, Q4, Q6, Q7, Q8, Q9",0,Not listed (Yes)


In [30]:
#@title 23. Compare with the study's stored MARCA scores and the expert gold standard (if the case is in the KB)
kb_case = kb_df[kb_df["case_id"].astype(str).str.strip() == str(result["case_id"]).strip()].copy()
comparison = pd.DataFrame()
if kb_case.empty:
    print(f"Case {result['case_id']} is not in the knowledge base: no expert reference to compare with.")
else:
    kb_by_key = {}
    for _, k in kb_case.iterrows():   # a DEC listed twice in the KB keeps its first row
        kb_by_key.setdefault((norm_drug(k["drug_name"]), norm_ae(k["pt"])), k)
    comp_rows = []
    for _, r in score_sheet.iterrows():
        key = (norm_drug(r["Drug"]), norm_ae(r["Adverse event"]))
        if key not in kb_by_key:
            continue
        k = kb_by_key[key]
        out = {"Drug": r["Drug"], "Adverse event": r["Adverse event"]}
        for q in ("Q1", "Q2", "Q5", "Q10"):
            out[f"{q} this run"] = r[q]
            out[f"{q} study MARCA"] = k[f"{q.lower()}_score_ai"]
            out[f"{q} expert"] = k[f"{q.lower()}_score_human"]
        out["Total this run"] = r["Total"]
        out["Total study MARCA"] = k["score_ai"]
        out["Total expert"] = k["score_human"]
        out["Category this run"] = r["Category"]
        out["Category study MARCA"] = naranjo_category(k["score_ai"])
        out["Category expert"] = naranjo_category(k["score_human"])
        comp_rows.append(out)
    comparison = pd.DataFrame(comp_rows)
    print("Note: this case's own records were excluded from retrieval (same case ID), as in the study.")
    display(comparison)
    if not comparison.empty:
        agree = (comparison["Category this run"] == comparison["Category expert"]).mean() * 100
        print(f"Category agreement with the expert for this case: {agree:.0f}% "
              f"({len(comparison)} DECs; Q3, Q4, Q6–Q9 as entered in the score sheet).")


Note: this case's own records were excluded from retrieval (same case ID), as in the study.


""


In [31]:
#@title 24. Save everything to Excel and download
xlsx_path = save_assessment_excel(result, extra_sheets={"Score_sheet": score_sheet,
                                                        "Expert_comparison": comparison})
try:
    from google.colab import files
    files.download(str(xlsx_path))
except Exception:
    print("Saved to", xlsx_path)


✅ Saved /content/marca/output/results/FDA_Case_229933175_Assessment.xlsx